# RAG Assignment: Baseline Retrieval vs Cross-Encoder Re-Ranking
**Primary corpus:** NIST Special Publication 800-30 Revision 1, *Guide for Conducting Risk Assessments* (September 2012, public domain)

Two pipelines over the **exact same chunked corpus**:
* **Pipeline 1 (baseline):** Bi-Encoder (all-MiniLM-L6-v2) -> cosine Top 10 -> first 3 chunks -> LLM (Answer 1)
* **Pipeline 2 (re-ranking):** the *same* Top 10 -> Cross-Encoder (ms-marco-MiniLM-L-6-v2) -> first 3 chunks -> LLM (Answer 2)

**Status of this notebook.** Corpus extraction, chunking, freezing and the pre-registered question set. Retrieval, re-ranking,
generation and latency are added in later stages, each after review.

**A pilot on a different corpus is preserved separately** in `pilot_catan/` (CATAN rulebook, 98 chunks): the Cross-Encoder produced
0 promotions and 0 demotions over 19 queries there. That pilot is a clearly labelled negative result and **none of its metrics
are combined with the NIST results below.** It influenced the design only in two disclosed ways: the chunking parameters were
carried over unchanged, and the question-writing protocol (pre-register, report every query) was kept.

In [1]:
import collections
import hashlib
import json
import os
import re
import statistics
import urllib.request
from pathlib import Path

# Keep every downloaded model file inside the (git-ignored) project folder. Must be set BEFORE huggingface_hub is imported.
os.environ.setdefault("HF_HOME", str(Path("data/hf_cache").resolve()))
# torch, faiss-cpu and scikit-learn each bundle their own libomp on macOS; with >1 OpenMP thread the kernel segfaults (seen in the
# pilot). One thread avoids it. All CPU timings in this project are therefore single-threaded.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pymupdf
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

# --- source document -----------------------------------------------------------------------------------------------------
PDF_URL = "https://nvlpubs.nist.gov/nistpubs/Legacy/SP/nistspecialpublication800-30r1.pdf"   # linked from csrc.nist.gov/pubs/sp/800/30/r1/final
PDF_SHA256 = "f214087f0bdb35932a28c16eb93932a33c67a6edfb5e6f1638866bd201c98e04"
DATA_DIR = Path("data/nist")
OUT_DIR = Path("results")
PDF_PATH = DATA_DIR / "nist_sp_800-30r1.pdf"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"      # its tokenizer measures chunk length (the model itself is used later)

# --- corpus scope (documented, applied uniformly; see the table audit below) -------------------------------------------------
FIRST_PAGE, LAST_PAGE = 9, 95            # PDF pages 1-8 = cover, legal notices, acknowledgements, contents, prologue (not content)
TEXT_REPAIRS = {"Arisk assessment may identify": "A risk assessment may identify"}   # Appendix J opens with the one-letter word "A" as a drop cap
EXCLUDE_PAGES = {49, 50, 64}             # Appendix A (reference list) and Appendix C (acronym list)
GLOSSARY_PAGES = set(range(51, 64))      # Appendix B: two-column term/definition list, parsed entry by entry

# --- chunking parameters: identical to the pilot, NOT tuned on NIST retrieval ---------------------------------------------------
MAX_CHUNK_TOKENS = 200      # soft cap per chunk (embedding-model tokenizer)
HARD_LIMIT_TOKENS = 240     # never exceed; stays under the Bi-Encoder's 256-token window
ENTRY_MAX_TOKENS = 240      # a glossary entry is atomic: allowed up to the hard limit
MIN_TOKENS = 30             # avoid a tiny trailing chunk inside a split section
OVERLAP_SENTENCES = 1
MIN_OVERLAP_TOKENS, MAX_OVERLAP_TOKENS = 8, 40
CHUNK_PREFIX = "nt"         # chunk ids are nt-001, nt-002, ... (the pilot used ck-)

# Pinned BEFORE any evaluation question was written. Freeze history (all three before any question or any retrieval existed;
# every change was a text-cleaning defect found by reading the extracted text, none was informed by retrieval results):
#   freeze 1  2026-10-18 00:37 IST  98b40d5e72eb...  superseded: words glued together (whitespace-only spans / the space after a
#                                                   footnote mark had been dropped)
#   freeze 2  2026-10-18 00:39 IST  8b22a428e402...  superseded: one glossary definition cut in two and one drop-cap word glued
#   freeze 3  2026-10-18 00:40 IST  795fb2a6e043...  FINAL. Changed 3 of 305 chunks (nt-242, nt-243, nt-291); ids, scope and all
#                                                   chunking parameters unchanged. Nothing may change from here on.
FROZEN_CORPUS_SHA256 = "795fb2a6e043a28dc39d01e2db30d4b05d74ca1535df4ca7519d386b15f0fc92"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

/Users/charan/Documents/LanguageModels&Agents/RAGAssignment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stage N1a. Fetch and verify the official PDF
The PDF is a US-government publication ("not subject to copyright protection in the United States"), so it and the chunk text
may be shared. The SHA-256 below was checked against a fresh download from the official NIST URL (byte-identical).

In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def ensure_pdf() -> Path:
    """Download the official PDF if absent, then verify its SHA-256 (fail loudly on mismatch)."""
    if not PDF_PATH.exists():
        print("Downloading", PDF_URL)
        req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
        PDF_PATH.write_bytes(urllib.request.urlopen(req, timeout=120).read())
    got = sha256_file(PDF_PATH)
    assert got == PDF_SHA256, f"PDF hash mismatch: expected {PDF_SHA256}, got {got}"
    return PDF_PATH


ensure_pdf()
doc = pymupdf.open(PDF_PATH)
print(f"PDF verified: {PDF_PATH}  sha256={PDF_SHA256[:16]}...  pages={len(doc)}  source={PDF_URL}")

PDF verified: data/nist/nist_sp_800-30r1.pdf  sha256=f214087f0bdb3593...  pages=95  source=https://nvlpubs.nist.gov/nistpubs/Legacy/SP/nistspecialpublication800-30r1.pdf


In [3]:
tok_file = hf_hub_download(EMBED_MODEL, "tokenizer.json")
cfg_file = hf_hub_download(EMBED_MODEL, "sentence_bert_config.json")
MODEL_MAX_SEQ = json.load(open(cfg_file))["max_seq_length"]
tok = Tokenizer.from_file(tok_file)
tok.no_truncation()
tok.no_padding()


def n_tokens(text: str) -> int:
    """Token count incl. [CLS]/[SEP]: what the Bi-Encoder actually sees."""
    return len(tok.encode(text).ids)


assert HARD_LIMIT_TOKENS <= MODEL_MAX_SEQ
print(f"{EMBED_MODEL}: max_seq_length={MODEL_MAX_SEQ} tokens; soft cap={MAX_CHUNK_TOKENS}, hard limit={HARD_LIMIT_TOKENS}")

sentence-transformers/all-MiniLM-L6-v2: max_seq_length=256 tokens; soft cap=200, hard limit=240


## Stage N1b. Table audit: why tables are outside the corpus scope
Appendices D-I are mostly tables (threat events, vulnerabilities, likelihood and impact scales, risk matrices) set in a narrow
sans-serif face in 3-6 grid columns. Reading such a page as text interleaves the columns, so a "row" is not a contiguous run of text.
The sample below is what plain text extraction returns for one table page; mixing this into the index would put scrambled
fragments next to clean prose. **Scope decision (uniform for the whole document):** only text set in the document's body typeface
(Times) is indexed. Text in tables, figures and diagrams (Arial / Arial Narrow / Calibri) is excluded, and so are footnotes
(found geometrically: text below Word's footnote separator rule) and running headers and footers. Appendix B (glossary) is a
two-column list and gets a dedicated parser that rebuilds each *term -> definition* entry.

In [4]:
sample_page = doc[69]            # PDF page 70, Appendix E table of adversarial threat events
txt = " ".join(sample_page.get_text().split())
print("PDF page 70, plain extraction (first 700 characters):\n")
print(txt[220:920])

PDF page 70, plain extraction (first 700 characters):

DVERSARIAL THREAT EVENTS54 Threat Events (Characterized by TTPs) Description Perform reconnaissance and gather information. Perform perimeter network reconnaissance/scanning. Adversary uses commercial or free software to scan organizational perimeters to obtain a better understanding of the information technology infrastructure and improve the ability to launch successful attacks. Perform network sniffing of exposed networks. Adversary with access to exposed wired or wireless data channels used to transmit information, uses network sniffing to identify components, resources, and protections. Gather information using open source discovery of organizational information. Adversary mines publica


## Stage N1c. Extraction and cleaning
Every text line is read with its font, size and position. Rules (all applied uniformly):
* **drop** running headers/footers (top/bottom margin), footnotes (below the separator rule), superscript note marks inside lines,
  and every non-body-typeface line (tables, figure labels, diagram boxes);
* **rebuild** bullets (the bullet glyph is a separate line) and drop caps (the first letter of a chapter is a 51-pt line);
* **headings** from typeface: chapter/appendix label+title, numbered sections `n.n`, STEP and TASK lines, short italic sub-heads;
* **paragraphs** from line spacing; paragraphs cut by a page break in the middle of a sentence are stitched;
* **glossary** pages: entry = term + definition (+ source), rebuilt from the two columns.
Every dropped line is logged with its reason.

In [5]:
TERMINAL = (".", "!", "?", ":", ";", ")", "”", '"')
GLOSS_TERM_X, GLOSS_DEF_X = (93, 98), (258, 266)


def read_page_lines(page, pno):
    d = page.get_text("dict")
    # Word's footnote separator: a short filled rule, left-aligned at the text margin (x = 90), about 144 pt wide.
    # (Table borders and task boxes are also thin rules, so the signature must be this specific.)
    rules = [dr["rect"] for dr in page.get_drawings() if dr["rect"].height <= 2.5 and 88 <= dr["rect"].x0 <= 92
             and 138 <= dr["rect"].width <= 150 and 100 < dr["rect"].y0 < 735]
    foot_y = rules[0].y0 if len(rules) == 1 else None            # several matching rules = table cell borders, not a footnote page
    lines = []
    for b in d["blocks"]:
        if b["type"] != 0:
            continue
        for l in b["lines"]:
            spans = [s for s in l["spans"] if s["text"].strip()]
            body = [s for s in spans if not (s["flags"] & 1)]          # superscript = footnote reference mark
            if not body:
                continue
            # keep whitespace-only spans (they are the spaces between words) and replace a footnote mark by a space, so words never glue
            text = re.sub(r"\s+", " ", "".join(" " if (s["flags"] & 1 and s["text"].strip()) else s["text"] for s in l["spans"])).strip()
            n_chars = sum(len(s["text"]) for s in body)
            x0, y0, x1, y1 = l["bbox"]
            lines.append(dict(
                page=pno, text=text, x0=x0, y0=y0, x1=x1, y1=y1, size=max(s["size"] for s in body),
                font=max(body, key=lambda s: len(s["text"]))["font"],
                italic=sum(len(s["text"]) for s in body if s["flags"] & 2) / n_chars > 0.6,
                foot=foot_y is not None and y0 >= foot_y - 1, bullet=False, why=None))
    # bullet glyphs and drop caps are separate lines: fold them into the line they belong to
    for L in [x for x in lines if x["text"] == "•"]:
        right = [x for x in lines if x is not L and x["x0"] > L["x0"] and abs(x["y0"] - L["y0"]) < 5 and x["text"] != "•"]
        if right:
            nxt = min(right, key=lambda x: x["x0"])
            nxt["bullet"] = True
            L["why"] = "merged bullet glyph"
    for L in [x for x in lines if x["size"] >= 30 and len(x["text"]) == 1 and x["text"].isalpha()]:
        below = [x for x in lines if x is not L and 0 < x["y0"] - L["y0"] < 40 and x["x0"] > L["x0"] and x["size"] < 20]
        if below:
            nxt = min(below, key=lambda x: (x["y0"], x["x0"]))
            nxt["text"] = L["text"] + nxt["text"]
            L["why"] = "merged drop cap"
    return lines


def classify(L, pno):
    """Return kind in {h1, h2, body, glossary_term, glossary_def, drop} and fill L['why'] for drops."""
    if L["why"]:
        return "drop"
    f, sz, t = L["font"], L["size"], L["text"]
    if L["y0"] < 52 or L["y0"] > 735:
        L["why"] = "running header / footer"
        return "drop"
    if L["foot"]:
        L["why"] = "footnote text"
        return "drop"
    if pno in GLOSSARY_PAGES and f.startswith("Times"):
        if GLOSS_TERM_X[0] <= L["x0"] <= GLOSS_TERM_X[1]:
            return "glossary_term"
        if GLOSS_DEF_X[0] <= L["x0"] <= GLOSS_DEF_X[1]:
            return "glossary_def"
    if pno == 9:                                                       # Cautionary Notes: Arial text is the content here
        if f.startswith("Arial-BoldItalic"):
            return "h1"
        if f == "ArialMT" and sz < 8:
            return "h1"
        if f == "ArialMT" or f.startswith("SymbolMT"):
            return "body"
    if f.startswith("ArialNarrow-Bold") and sz >= 12.5:
        return "h1"
    if f.startswith("Arial-BoldMT") and sz >= 14:
        return "h1"
    if f == "ArialMT" and sz == 9.0 and t.isupper() and L["y0"] < 140:
        return "h1"
    if f.startswith("Arial-BoldMT") and 10.5 <= sz < 12.5:                       # numbered sections "n.n TITLE" (set in small caps, 11-12 pt)
        return "h2"
    if f.startswith("ArialNarrow-Bold") and 10.5 <= sz < 12.5 and (re.match(r"STEP \d", t) or pno >= 90):
        return "h2"          # in Chapters 1-3 other ArialNarrow-Bold 11 text is diagram labels; Appendix J-L use it for sub-headings
    if f.startswith("Times"):
        return "body"
    L["why"] = "table / figure / diagram text (non-body typeface)"
    return "drop"


def build_paragraphs(lines, pno):
    lines = sorted(lines, key=lambda L: (round(L["y0"]), L["x0"]))
    paras, cur, prev = [], None, None
    for L in lines:
        kind = L["kind"]
        is_task = bool(re.match(r"TASK \d-\d", L["text"]))
        starts_block = prev is None or L["y0"] - prev["y0"] > 1.3 * L["size"]
        if kind == "body" and L["italic"] and len(L["text"].split()) <= 8 and not L["text"].endswith((".", ":", ";", ",")) \
                and not L["bullet"] and not is_task and starts_block:
            kind = "h3"
        if is_task:
            kind = "h3"
        continues_task = (cur is not None and cur["kind"] == "h3" and re.match(r"TASK \d-\d", cur["lines"][0]["text"]) and kind == "body"
                          and not L["bullet"] and abs(L["size"] - cur["size"]) < 0.4 and 0 < L["y0"] - prev["y0"] <= 1.6 * L["size"])
        if continues_task:
            kind = "h3"
        new = cur is None or kind != cur["kind"] or kind == "h3" and not continues_task and (is_task or L["y0"] - prev["y0"] > 1.6 * L["size"])
        if not new and kind in ("h1", "h2"):
            new = L["y0"] - prev["y0"] > 2.2 * L["size"] or (kind == "h2" and bool(re.match(r"\d\.\d", L["text"])))
        if not new and kind == "body":
            new = (L["bullet"] or L["text"].startswith("•") or L["y0"] - prev["y0"] > 1.6 * L["size"]
                   or abs(L["size"] - prev["size"]) > 0.4 or L["y0"] < prev["y0"])
        if new:
            cur = dict(kind=kind, lines=[L], size=L["size"], page=pno)
            paras.append(cur)
        else:
            cur["lines"].append(L)
        prev = L
    for p in paras:
        sep = " — " if p["kind"] == "h1" else " "
        t = sep.join(L["text"] for L in p["lines"])
        t = re.sub(r"\s+", " ", t).strip()
        t = re.sub(r"(?<=[a-z])- (?!(?:or|and|nor)\b)(?=[a-z])", "-", t)        # line-break hyphen repair; keep suspended hyphens ("hurricane- or flood-prone")
        t = re.sub(r"^(\d\.\d)(?=[A-Z])", r"\1 ", t)
        t = re.sub(r"^(Supplemental Guidance:)(?=\S)", r"\1 ", t)
        t = re.sub(r"^•(?=\S)", "• ", t)
        for bad, good in TEXT_REPAIRS.items():
            t = t.replace(bad, good)
        t = ("• " + t) if p["lines"][0]["bullet"] and not t.startswith("•") else t
        p["text"] = t
    return paras


def extract_glossary(lines, pno, state):
    """Rebuild glossary entries from the two columns. `state` carries the open entry across pages."""
    entries = state["entries"]
    term_rows = [L for L in lines if L["kind"] == "glossary_term" and not (L["text"].startswith("[") or abs(L["size"] - 10.0) < 0.15)]
    for d in (L for L in lines if L["kind"] == "glossary_def"):
        for t in term_rows:                                    # a definition's first line can sit up to ~3 pt ABOVE its term's line
            if 0 <= t["y0"] - d["y0"] <= 4.5:
                d["y0"] = t["y0"]
                break
    for L in sorted(lines, key=lambda L: (round(L["y0"]), L["x0"])):
        if L["kind"] == "glossary_term":
            if L["text"].startswith("[") or abs(L["size"] - 10.0) < 0.15:
                if entries:
                    entries[-1]["source"] = (entries[-1]["source"] + " " + L["text"]).strip()
            elif entries and pno == entries[-1]["page"] and 0 < L["y0"] - state["last_term_y"] <= 15:
                entries[-1]["term"] += " " + L["text"]       # wrapped term: its 2nd/3rd line sits one line-pitch (~12.5 pt) below, a new entry >= 18 pt
                state["last_term_y"] = L["y0"]
            else:
                entries.append(dict(term=L["text"], definition="", source="", page=pno))
                state["last_term_y"] = L["y0"]
        elif L["kind"] == "glossary_def" and entries:
            entries[-1]["definition"] = (entries[-1]["definition"] + " " + L["text"]).strip()


def extract_document(doc):
    paras, dropped = [], []
    gstate = dict(entries=[], last_term_y=-99)
    for pno in range(1, len(doc) + 1):
        page = doc[pno - 1]
        lines = read_page_lines(page, pno)
        if pno < FIRST_PAGE or pno in EXCLUDE_PAGES:
            for L in lines:
                if not L["why"]:
                    L["why"] = "page outside the corpus scope" if pno in EXCLUDE_PAGES else "front matter (pages 1-8)"
                    L["kind"] = "drop"
                dropped.append(L)
            continue
        for L in lines:
            L["kind"] = classify(L, pno)
        dropped += [L for L in lines if L["kind"] == "drop"]
        keep = [L for L in lines if L["kind"] != "drop"]
        if pno in GLOSSARY_PAGES:
            extract_glossary([L for L in keep if L["kind"].startswith("glossary")], pno, gstate)
            keep = [L for L in keep if not L["kind"].startswith("glossary")]
        paras += build_paragraphs(keep, pno)
        if pno in GLOSSARY_PAGES and pno == max(GLOSSARY_PAGES):
            paras.append(dict(kind="glossary_block", entries=gstate["entries"], page=min(GLOSSARY_PAGES), text="", size=11))
    return paras, dropped


paragraphs, dropped_lines = extract_document(doc)

# stitch paragraphs that a page break cut in the middle of a sentence
stitched, n_stitched = [], 0
for p in paragraphs:
    prev = stitched[-1] if stitched else None
    if (prev and prev["kind"] == "body" and p["kind"] == "body" and p["text"][:1].islower() and not p["lines"][0]["bullet"]
            and not prev["text"].rstrip().endswith(TERMINAL) and p["page"] != prev["page"]):
        prev["text"] += " " + p["text"]
        n_stitched += 1
    else:
        stitched.append(p)
paragraphs = stitched
print(f"paragraph blocks: {len(paragraphs)}   stitched across page breaks: {n_stitched}")

paragraph blocks: 319   stitched across page breaks: 10


In [6]:
raw_words = sum(len(doc[i].get_text().split()) for i in range(FIRST_PAGE - 1, LAST_PAGE) if (i + 1) not in EXCLUDE_PAGES)
by_reason = collections.Counter()
for L in dropped_lines:
    if L["page"] >= FIRST_PAGE and L["page"] not in EXCLUDE_PAGES:
        by_reason[L["why"]] += len(L["text"].split())
body_words = sum(len(p["text"].split()) for p in paragraphs if p["kind"] in ("body", "h1", "h2", "h3"))
gloss = next(p for p in paragraphs if p["kind"] == "glossary_block")["entries"]
gloss_words = sum(len((e["term"] + " " + e["definition"] + " " + e["source"]).split()) for e in gloss)
print(f"in-scope pages: {FIRST_PAGE}-{LAST_PAGE} minus {sorted(EXCLUDE_PAGES)}   raw words on those pages (PyMuPDF get_text): {raw_words:,}")
print("words removed inside the in-scope pages, by reason:")
for r, n in by_reason.most_common():
    print(f"   {n:6,d}  {r}")
print(f"words kept: narrative {body_words:,} + glossary {gloss_words:,} (in {len(gloss)} entries) = {body_words + gloss_words:,}")
print("front matter / excluded pages dropped:", f"{sum(len(L['text'].split()) for L in dropped_lines if L['page'] < FIRST_PAGE or L['page'] in EXCLUDE_PAGES):,} words")

in-scope pages: 9-95 minus [49, 50, 64]   raw words on those pages (PyMuPDF get_text): 39,540
words removed inside the in-scope pages, by reason:
   11,412  table / figure / diagram text (non-body typeface)
    2,469  footnote text
    1,090  running header / footer
      105  merged bullet glyph
       12  merged drop cap
words kept: narrative 20,261 + glossary 4,252 (in 119 entries) = 24,513
front matter / excluded pages dropped: 2,844 words


## Stage N1d. Audit of the cleaned text (samples)

In [7]:
def show(page, contains, max_words=110):
    for p in paragraphs:
        if p["page"] == page and p["kind"] == "body" and contains in p["text"]:
            w = p["text"].split()
            print(" ".join(w[:max_words]) + (" ..." if len(w) > max_words else ""), "\n")
            return
    print("(not found)", page, contains)


print("--- Chapter 2 body paragraph, PDF page 20 (originally interrupted by a footnote at the page foot) ---")
show(20, "three-step process")
print("--- Chapter 3: TASK statement (heading) and the first paragraph of its supplemental guidance (9-pt text; footnotes excluded), PDF page 33 ---")
i = next(i for i, p in enumerate(paragraphs) if p["kind"] == "h3" and p["text"].startswith("TASK 1-1"))
print("[task heading]", paragraphs[i]["text"])
print("[guidance]    ", " ".join(paragraphs[i + 1]["text"].split()[:70]) + " ...")
print("\n--- Appendix B glossary entry rebuilt from two columns ---")
e = next(e for e in gloss if e["term"].lower().startswith("threat source"))
print(f"{e['term']}: {e['definition']} {e['source']}")
print("\n--- Appendix E opening paragraph (drop cap re-joined; the table below it is out of scope) ---")
show(69, "This appendix provides", 60)

--- Chapter 2 body paragraph, PDF page 20 (originally interrupted by a footnote at the page foot) ---
Organizations typically employ a three-step process to determine the overall likelihood of threat events. First, organizations assess the likelihood that threat events will be initiated (for adversarial threat events) or will occur (for non-adversarial threat events). Second, organizations assess the likelihood that the threat events once initiated or occurring, will result in adverse impacts or harm to organizational operations and assets, individuals, other organizations, or the Nation. Finally, organizations assess the overall likelihood as a combination of likelihood of initiation/occurrence and likelihood of resulting in adverse impact. 

--- Chapter 3: TASK statement (heading) and the first paragraph of its supplemental guidance (9-pt text; footnotes excluded), PDF page 33 ---
[task heading] TASK 1-1: Identify the purpose of the risk assessment in terms of the information that th

## Stage N2a. Chunking (structure-aware; parameters identical to the pilot)
* A **section** is the text under one heading (a numbered section, a `TASK n-n` statement, an italic sub-head, ...). **Chunks never
  cross a heading.** The heading text stays at the start of the section's first chunk, as in the document.
* Paragraphs are packed greedily up to `MAX_CHUNK_TOKENS`; a paragraph is split (at sentence boundaries) only if it alone exceeds the cap.
* A continuation chunk repeats the last sentence of the previous one (overlap), unless it is very short or long.
* Every **glossary entry** (term + definition + source) is one chunk of its own, up to the hard limit.
* Metadata (not embedded): `chunk_id`, pages, `section`, glossary `entry`, token count, text hash.

In [8]:
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])[\"”’)]?\s+(?=[\"“(•A-Z0-9])")


def split_sentences(text: str):
    parts, start = [], 0
    for m in SENT_SPLIT_RE.finditer(text):
        parts.append(text[start:m.end()].strip())
        start = m.end()
    parts.append(text[start:].strip())
    return [s for s in parts if s]


def split_oversized(block: str, cap: int):                 # returns text pieces
    pieces, cur = [], []
    for s in split_sentences(block):
        if cur and n_tokens(" ".join(cur + [s])) > cap:
            pieces.append(" ".join(cur))
            cur = []
        cur.append(s)
    if cur:
        pieces.append(" ".join(cur))
    return pieces


def chunk_section(blocks, cap=MAX_CHUNK_TOKENS):
    """Pack the (text, page) blocks of ONE section into chunks of at most ~cap tokens.
    Returns [(text, has_overlap, first_page, last_page)]; the pages are those of the chunk's own paragraphs."""
    pieces = []                                            # (text, page)
    for text, pg in blocks:
        for t in ([text] if n_tokens(text) <= cap else split_oversized(text, cap)):
            pieces.append((t, pg))
    chunks, cur, cur_overlap = [], [], False
    emit = lambda: chunks.append(("\n".join(t for t, _ in cur), cur_overlap, min(p for _, p in cur), max(p for _, p in cur)))
    for i, (piece, pg) in enumerate(pieces):
        candidate = "\n".join([t for t, _ in cur] + [piece])
        if cur and n_tokens(candidate) > cap:
            remaining = n_tokens("\n".join(t for t, _ in pieces[i:]))
            if remaining < MIN_TOKENS and n_tokens(candidate + "\n" + "\n".join(t for t, _ in pieces[i + 1:])) <= HARD_LIMIT_TOKENS:
                cur.append((piece, pg))                    # absorb a tiny tail instead of emitting a stub chunk
                continue
            emit()
            last_text, last_pg = cur[-1]
            last_sentence = split_sentences(last_text)[-1] if OVERLAP_SENTENCES else ""
            if last_sentence and MIN_OVERLAP_TOKENS <= n_tokens(last_sentence) <= MAX_OVERLAP_TOKENS and \
                    n_tokens(last_sentence + "\n" + piece) <= cap:
                cur, cur_overlap = [(last_sentence, last_pg), (piece, pg)], True
            else:
                cur, cur_overlap = [(piece, pg)], False
        else:
            cur.append((piece, pg))
    if cur:
        emit()
    return chunks


def short_label(text):
    m = re.match(r"(TASK \d-\d)", text)
    return m.group(1) if m else text


def build_chunks(paras):
    h1 = h2 = h3 = None
    sections, cur, pending = [], None, []
    for p in paras:
        k = p["kind"]
        if k == "glossary_block":
            for e in p["entries"]:
                body = f"{e['term']}\n{e['definition']} {e['source']}".strip()
                sections.append(dict(h1=h1, h2=e["term"], h3=None, heads=[], blocks=[(body, e["page"])], hpages=[], entry=e["term"]))
            cur = None
            continue
        if k in ("h1", "h2", "h3"):
            if k == "h1":
                h1 = (h1 + " — " + p["text"]) if pending and pending[-1]["kind"] == "h1" else p["text"]
                h2 = h3 = None
            elif k == "h2":
                h2, h3 = p["text"], None
            else:
                h3 = p["text"]
            pending.append(p)
            cur = None
            continue
        if cur is None:
            cur = dict(h1=h1, h2=h2, h3=h3, heads=[x["text"] for x in pending], hpages=[x["page"] for x in pending], blocks=[], entry=None)
            pending = []
            sections.append(cur)
        cur["blocks"].append((p["text"], p["page"]))
    chunks = []
    for sec in sections:
        blocks = list(sec["blocks"])
        if sec["heads"]:                                                # headings stay in the text, as in the document
            blocks[0] = ("\n".join(sec["heads"] + [blocks[0][0]]), min([blocks[0][1]] + sec["hpages"]))
        name = " > ".join(x for x in (sec["h1"], sec["h2"], short_label(sec["h3"]) if sec["h3"] else None) if x)
        for text, has_overlap, p_first, p_last in chunk_section(blocks, ENTRY_MAX_TOKENS if sec["entry"] else MAX_CHUNK_TOKENS):
            chunks.append(dict(text=text, section=name, entry=sec["entry"], page_start=p_first, page_end=p_last, has_overlap=has_overlap))
    for i, c in enumerate(chunks, 1):
        c["chunk_id"] = f"{CHUNK_PREFIX}-{i:03d}"
        c["n_tokens"] = n_tokens(c["text"])
        c["text_sha256"] = hashlib.sha256(c["text"].encode("utf-8")).hexdigest()
    return chunks


CHUNKS = build_chunks(paragraphs)

## Stage N2b. Chunking statistics and invariants

In [9]:
toks = [c["n_tokens"] for c in CHUNKS]
gl_chunks = [c for c in CHUNKS if c["entry"]]
nar_chunks = [c for c in CHUNKS if not c["entry"]]
q = statistics.quantiles(toks, n=10)
print(f"chunks: {len(CHUNKS)}  (narrative: {len(nar_chunks)}, glossary entries: {len(gl_chunks)} -> {len({c['entry'] for c in gl_chunks})} distinct terms)")
print(f"tokens/chunk  min={min(toks)}  p10={q[0]:.0f}  median={statistics.median(toks):.0f}  mean={statistics.mean(toks):.0f}  p90={q[-1]:.0f}  max={max(toks)}"
      f"   (soft cap {MAX_CHUNK_TOKENS}, hard limit {HARD_LIMIT_TOKENS}, model window {MODEL_MAX_SEQ})")
for name, grp in (("narrative", nar_chunks), ("glossary", gl_chunks)):
    t = [c["n_tokens"] for c in grp]
    print(f"   {name:9s} n={len(t):3d}  min={min(t)}  median={statistics.median(t):.0f}  max={max(t)}")
hist = collections.Counter(min(t // 40 * 40, 240) for t in toks)
print("token-length histogram (bin start: chunks):", {f"{b}-{b + 39}" if b < 240 else "240+": hist[b] for b in sorted(hist)})
print(f"chunks under {MIN_TOKENS} tokens: {sum(t < MIN_TOKENS for t in toks)}   over soft cap: {sum(t > MAX_CHUNK_TOKENS for t in toks)}   "
      f"with overlap prefix: {sum(c['has_overlap'] for c in CHUNKS)}")
print(f"words in chunks (incl. repeated overlap sentences): {sum(len(c['text'].split()) for c in CHUNKS):,}")
secs = collections.Counter((c["section"].split(' > ')[0][:34]) for c in nar_chunks)
print("narrative chunks per top-level part:", dict(secs))

assert max(toks) <= HARD_LIMIT_TOKENS <= MODEL_MAX_SEQ, "a chunk would be truncated by the Bi-Encoder"
assert len({c["chunk_id"] for c in CHUNKS}) == len(CHUNKS) and all(c["text"].strip() for c in CHUNKS)
print("invariants OK: unique ids, no empty chunk, every chunk fits the Bi-Encoder window")

chunks: 305  (narrative: 186, glossary entries: 119 -> 119 distinct terms)


tokens/chunk  min=7  p10=32  median=109  mean=112  p90=191  max=226   (soft cap 200, hard limit 240, model window 256)
   narrative n=186  min=31  median=166  max=226
   glossary  n=119  min=7  median=46  max=186
token-length histogram (bin start: chunks): {'0-39': 46, '40-79': 77, '80-119': 37, '120-159': 39, '160-199': 99, '200-239': 7}
chunks under 30 tokens: 24   over soft cap: 4   with overlap prefix: 45
words in chunks (incl. repeated overlap sentences): 25,423
narrative chunks per top-level part: {'CAUTIONARY NOTES — SCOPE AND APPLI': 3, 'CHAPTER ONE — INTRODUCTION — THE N': 13, 'CHAPTER TWO — THE FUNDAMENTALS — B': 76, 'CHAPTER THREE — THE PROCESS — COND': 72, 'APPENDIX B — GLOSSARY — COMMON TER': 1, 'APPENDIX D — THREAT SOURCES — TAXO': 1, 'APPENDIX E — THREAT EVENTS — REPRE': 1, 'APPENDIX F — VULNERABILITIES AND P': 1, 'APPENDIX G — LIKELIHOOD OF OCCURRE': 1, 'APPENDIX H — IMPACT — EFFECTS OF T': 1, 'APPENDIX I — RISK DETERMINATION — ': 1, 'APPENDIX J — INFORMING RISK RESPON

## Stage N2c. Freeze the corpus
From here on every pipeline reads the same chunks from disk and re-checks a SHA-256 over every (chunk_id, text) pair.
`FROZEN_CORPUS_SHA256` in the configuration cell was set from this printout **before any evaluation question was written**.

In [10]:
def corpus_hash(chunks) -> str:
    h = hashlib.sha256()
    for c in chunks:
        h.update(c["chunk_id"].encode() + b"\x00" + c["text"].encode("utf-8") + b"\x01")
    return h.hexdigest()


CORPUS_SHA256 = corpus_hash(CHUNKS)
if FROZEN_CORPUS_SHA256 is not None:
    assert CORPUS_SHA256 == FROZEN_CORPUS_SHA256, "extraction/chunking changed: the corpus is no longer the frozen one"
with open(DATA_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in CHUNKS:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
(OUT_DIR / "nist_chunks.jsonl").write_text("".join(json.dumps(c, ensure_ascii=False) + "\n" for c in CHUNKS), encoding="utf-8")  # public-domain text

manifest = dict(
    source=dict(title="NIST SP 800-30 Rev. 1, Guide for Conducting Risk Assessments (Sept. 2012)", url=PDF_URL, sha256=PDF_SHA256, pages=len(doc)),
    scope=dict(first_page=FIRST_PAGE, last_page=LAST_PAGE, excluded_pages=sorted(EXCLUDE_PAGES), glossary_pages=[min(GLOSSARY_PAGES), max(GLOSSARY_PAGES)],
               rules="body typeface only; tables/figures/footnotes/headers excluded; glossary parsed per entry"),
    parameters=dict(embed_model_tokenizer=EMBED_MODEL, model_max_seq_length=MODEL_MAX_SEQ, max_chunk_tokens=MAX_CHUNK_TOKENS,
                    hard_limit_tokens=HARD_LIMIT_TOKENS, entry_max_tokens=ENTRY_MAX_TOKENS, min_tokens=MIN_TOKENS,
                    overlap_sentences=OVERLAP_SENTENCES, extractor=f"PyMuPDF {pymupdf.__version__}"),
    corpus_sha256=CORPUS_SHA256, n_chunks=len(CHUNKS),
    chunks=[{k: c[k] for k in ("chunk_id", "section", "entry", "page_start", "page_end", "n_tokens", "has_overlap", "text_sha256")} for c in CHUNKS])
(OUT_DIR / "nist_corpus_manifest.json").write_text(json.dumps(manifest, indent=1, ensure_ascii=False))


def load_frozen_chunks():
    chunks = [json.loads(line) for line in open(DATA_DIR / "chunks.jsonl", encoding="utf-8")]
    expected = json.load(open(OUT_DIR / "nist_corpus_manifest.json"))["corpus_sha256"]
    assert corpus_hash(chunks) == expected, "corpus changed since it was frozen"
    return chunks


assert load_frozen_chunks() == CHUNKS
print(f"corpus_sha256 = {CORPUS_SHA256}" + ("   (pinned: matches)" if FROZEN_CORPUS_SHA256 else "   (NOT pinned yet)"))
print(f"{len(CHUNKS)} chunks written to data/nist/chunks.jsonl and results/nist_chunks.jsonl; manifest in results/nist_corpus_manifest.json")

corpus_sha256 = 795fb2a6e043a28dc39d01e2db30d4b05d74ca1535df4ca7519d386b15f0fc92   (pinned: matches)
305 chunks written to data/nist/chunks.jsonl and results/nist_chunks.jsonl; manifest in results/nist_corpus_manifest.json


---
# Stage N3. Pre-registered evaluation questions (written after the corpus was frozen, before any retrieval)
**Protocol**
* 24 questions = 6 each of the four failure categories in the assignment: **keyword trap / distractor**, **exception located elsewhere**,
  **negation / constraint**, and **other buried detail**. They were written after reading the frozen chunks and *before any question
  was embedded or run against any index* (no embedding of this corpus even exists yet).
* Each question has one or more **evidence phrases**. A phrase is resolved to the chunk(s) that contain it by exact string match,
  and the resolved gold set is compared with the set I expected when writing the question. Any chunk that really contains the answer
  is listed; a question counts as answered by the baseline if **any** gold chunk is in the Top 3.
* `likely_distractors` are my *predictions* of competing chunks, recorded in advance so they can be checked against the results.
* The list, its resolved gold labels and the corpus hash are hashed into `PREREG_SHA256`, which is pinned in this notebook.
  Editing a question or a label after seeing results would stop the notebook.
* **Every** question is reported later (k of N), including the ones the baseline answers correctly.
* Disclosure: the CATAN pilot (separate notebook) showed that questions reusing the document's own wording are easy for the Bi-Encoder
  and that fully paraphrased ones can fall out of the Top 10 altogether. I therefore kept some domain terms in every question while
  shifting the emphasis or the wording. No NIST retrieval result influenced any question.

In [11]:
PREREG = [
    # ---- A. keyword trap / distractor ------------------------------------------------------------------------------------------
    dict(id="A1", category="keyword trap", expect=["nt-063"],
         question="Which risk assessment orientation begins with the consequences we care about and our most critical assets, rather than with attackers or weaknesses?",
         evidence=[["an asset/impact-oriented approach starts with the identification of impacts or consequences of concern and critical assets"]],
         likely_distractors=["nt-064", "nt-117", "nt-124"]),
    dict(id="A2", category="keyword trap", expect=["entry:Reproducibility"],
         question="Which term describes different experts arriving at the same results from the same data?",
         evidence=[["the ability of different experts to produce the same results from the same data"]],
         likely_distractors=["entry:Repeatability", "nt-030", "nt-060"]),
    dict(id="A3", category="keyword trap", expect=["entry:Compensating Security Control"],
         question="What do you call a safeguard used in place of a recommended baseline control that still gives comparable protection?",
         evidence=[["employed by an organization in lieu of a recommended security control"]],
         likely_distractors=["entry:Common Control", "entry:Tailoring"]),
    dict(id="A4", category="keyword trap", expect=["nt-040"],
         question="Can something about an organization's environment, such as a system with no outside network connection, make a successful attack less likely?",
         evidence=[["stand-alone information system with no external network connectivity"]],
         likely_distractors=["nt-041", "nt-038", "nt-037"]),
    dict(id="A5", category="keyword trap", expect=["nt-043", "nt-044"],
         question="How is the chance that an event actually causes harm different from the chance that someone starts it?",
         evidence=[["regardless of the magnitude of harm that can be expected"]],
         likely_distractors=["nt-042", "entry:Likelihood of Occurrence", "nt-145"]),
    dict(id="A6", category="keyword trap", expect=["nt-035", "entry:Threat Shifting"],
         question="What is it called when adversaries change their behaviour after noticing our safeguards?",
         evidence=[["threat shifting is the response of adversaries to perceived safeguards"], ["response from adversaries to perceived safeguards"]],
         likely_distractors=["entry:Advanced Persistent Threat", "nt-036"]),
    # ---- E. exceptions and corner cases ----------------------------------------------------------------------------------------
    dict(id="E1", category="exception", expect=["nt-053", "nt-054", "nt-153"],
         question="Is the level of risk always capped by the level of impact?",
         evidence=[["upper bound for risk may fail to apply"], ["impact as an upper bound on risk may not hold"]],
         likely_distractors=["nt-051", "nt-152", "entry:Risk"]),
    dict(id="E2", category="exception", expect=["nt-146", "nt-147"],
         question="If a threat event has no identified weaknesses that could be exploited, do we keep analysing it in the current assessment?",
         evidence=[["no further consideration during the current assessment is warranted"]],
         likely_distractors=["nt-145", "nt-136", "nt-137"]),
    dict(id="E3", category="exception", expect=["nt-286"],
         question="Can a threat event be dropped from the analysis when no adversary is known to have the ability to carry it out?",
         evidence=[["eliminate certain threat events from further consideration if no adversary with the necessary capability has been identified"]],
         likely_distractors=["nt-145", "nt-136", "nt-135"]),
    dict(id="E4", category="exception", expect=["nt-163", "nt-164"],
         question="When conditions have changed since the last assessment, must every task be redone?",
         evidence=[["determine whether all tasks in the risk assessment process need to be repeated"], ["identifying and assessing only how selected risk factors have changed"]],
         likely_distractors=["nt-159", "nt-160", "nt-023"]),
    dict(id="E5", category="exception", expect=["nt-105"],
         question="Can the scope of a system-level assessment ignore weaknesses that come from controls the system inherits?",
         evidence=[["without consideration of inherited vulnerabilities"]],
         likely_distractors=["nt-101", "nt-102", "nt-139"]),
    dict(id="E6", category="exception", expect=["nt-076", "nt-077"],
         question="What extra step may be needed when rolling up mission-level results in an organization whose missions differ a lot?",
         evidence=[["expert analysis may be needed to normalize the results from tier 2 risk assessments"]],
         likely_distractors=["nt-075", "nt-078", "nt-079"]),
    # ---- N. negation and constraints -------------------------------------------------------------------------------------------
    dict(id="N1", category="negation/constraint", expect=["nt-126"],
         question="Does the guide prescribe a formula for combining semi-quantitative factor values?",
         evidence=[["does not specify algorithms for combining semi-quantitative values"]],
         likely_distractors=["nt-147", "nt-148", "nt-125"]),
    dict(id="N2", category="negation/constraint", expect=["nt-002"],
         question="Are there fixed requirements for how formal or detailed a risk assessment has to be?",
         evidence=[["there are no specific requirements with regard to"]],
         likely_distractors=["nt-118", "nt-131", "nt-003"]),
    dict(id="N3", category="negation/constraint", expect=["nt-131"],
         question="Is it mandatory to use the sample tables in the appendices when carrying out an assessment?",
         evidence=[["the use of the templates is not required in order to conduct risk assessments"]],
         likely_distractors=["nt-133", "nt-137", "nt-092"]),
    dict(id="N4", category="negation/constraint", expect=["nt-161", "nt-162"],
         question="Is watching only the security posture of the information systems enough to know the organization's overall risk?",
         evidence=[["monitoring only the security posture of information systems would likely not provide sufficient information"]],
         likely_distractors=["nt-160", "nt-159", "nt-091"]),
    dict(id="N5", category="negation/constraint", expect=["nt-010"],
         question="Does this guidance cover systems that are designated as national security systems?",
         evidence=[["other than those systems designated as national security systems"]],
         likely_distractors=["entry:National Security System", "nt-008", "nt-014"]),
    dict(id="N6", category="negation/constraint", expect=["nt-145"],
         question="Should we expect an adversary to attack an organization it is not actively targeting?",
         evidence=[["if adversaries are not actively targeting specific organizations or their missions/business functions, adversaries are not expected to initiate threat events"]],
         likely_distractors=["nt-144", "nt-146", "nt-042"]),
    # ---- O. other: buried detail in a long, list-like or very short chunk ------------------------------------------------------------
    dict(id="O1", category="other (buried detail)", expect=["nt-018", "entry:Risk Management"],
         question="What are the four components of the risk management process?",
         evidence=[["(i) framing risk; (ii) assessing risk; (iii) responding to risk; and (iv) monitoring risk"],
                   ["(i) establishing the context for risk-related activities; (ii) assessing risk; (iii) responding to risk once determined; and (iv) monitoring risk over time"]],
         likely_distractors=["nt-019", "nt-020", "nt-021"]),
    dict(id="O2", category="other (buried detail)", expect=["nt-061"],
         question="What example score ranges does the guide give for binning semi-quantitative values?",
         evidence=[["0-15, 16-35, 36-70, 71-85, 86-100"]],
         likely_distractors=["nt-060", "nt-058", "nt-062"]),
    dict(id="O3", category="other (buried detail)", expect=["nt-121", "nt-122"],
         question="Which external organizations does the guide name as sources of threat information?",
         evidence=[["us computer emergency readiness team"], ["information sharing and analysis centers"]],
         likely_distractors=["nt-123", "nt-120", "nt-119"]),
    dict(id="O4", category="other (buried detail)", expect=["nt-080"],
         question="At what point in the system development life cycle should an initial risk assessment ideally be performed?",
         evidence=[["ideally these assessments should be performed in the initiation phase"]],
         likely_distractors=["nt-023", "nt-081", "nt-082"]),
    dict(id="O5", category="other (buried detail)", expect=["nt-301"],
         question="What should the executive summary of a risk assessment report say about the overall level of risk?",
         evidence=[["describe the overall level of risk"]],
         likely_distractors=["nt-299", "nt-302", "nt-304"]),
    dict(id="O6", category="other (buried detail)", expect=["nt-125"],
         question="How many assessment scales does the guide provide for each risk factor, and of what kinds?",
         evidence=[["three assessment scales (one qualitative and two semi-quantitative scales)"]],
         likely_distractors=["nt-124", "nt-126", "nt-127"]),
]
PREREG_SHA256 = "cb7a05a13b16fd0aacbe310e7903df4f5ee34137d3cac3ebf85714ec197ace33"   # fixed 2026-10-18 00:43 IST, before any question was embedded or run


def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s.replace("“", '"').replace("”", '"').replace("’", "'")).strip().lower()


CHUNK_BY_ID = {c["chunk_id"]: c for c in CHUNKS}
ENTRY_TO_ID = {c["entry"]: c["chunk_id"] for c in CHUNKS if c["entry"]}


def to_id(ref):                       # "entry:Term" -> that glossary chunk id
    return ENTRY_TO_ID[ref[6:]] if ref.startswith("entry:") else ref


def resolve_gold(evidence):
    gold = set()
    for ev in evidence:
        hits = [c["chunk_id"] for c in CHUNKS if all(norm(p) in norm(c["text"]) for p in ev)]
        assert hits, f"evidence phrase not found in the frozen corpus: {ev}"
        gold |= set(hits)
    return sorted(gold)


PREREG_Q = []
for q in PREREG:
    gold = resolve_gold(q["evidence"])
    expected = sorted(to_id(e) for e in q["expect"])
    assert gold == expected, f"{q['id']}: resolved gold {gold} differs from the expectation {expected}"
    PREREG_Q.append(dict(id=q["id"], category=q["category"], question=q["question"], evidence=q["evidence"], gold_chunk_ids=gold,
                         likely_distractors=[to_id(d) for d in q["likely_distractors"]]))

canon = json.dumps(dict(corpus_sha256=CORPUS_SHA256, queries=PREREG_Q), sort_keys=True, ensure_ascii=False)
prereg_hash = hashlib.sha256(canon.encode()).hexdigest()
by_cat = collections.Counter(q["category"] for q in PREREG_Q)
print(f"questions: {len(PREREG_Q)}  by category: {dict(by_cat)}")
print(f"gold chunks per question: min={min(len(q['gold_chunk_ids']) for q in PREREG_Q)} max={max(len(q['gold_chunk_ids']) for q in PREREG_Q)}; "
      f"distinct gold chunks: {len({g for q in PREREG_Q for g in q['gold_chunk_ids']})}")
for q in PREREG_Q:
    pg = ", ".join(f"{g}(p{CHUNK_BY_ID[g]['page_start']}, {CHUNK_BY_ID[g]['n_tokens']}t)" for g in q["gold_chunk_ids"])
    print(f"  {q['id']} [{q['category'][:12]:12s}] gold: {pg}")
print(f"pre-registration hash (recomputed here) = {prereg_hash}")
if PREREG_SHA256 is None:
    print("(not pinned yet)")
else:
    print(f"pre-registration hash pinned in code    = {PREREG_SHA256}")
    assert prereg_hash == PREREG_SHA256, "the question set or its gold labels changed after pre-registration"
    print("PASS: questions and gold labels are exactly the ones registered before any retrieval was run")
(OUT_DIR / "nist_preregistration.json").write_text(json.dumps(dict(
    prereg_sha256=prereg_hash, corpus_sha256=CORPUS_SHA256,
    note="authored after the corpus was frozen and before any question was embedded or run against any index", queries=PREREG_Q),
    indent=1, ensure_ascii=False))

questions: 24  by category: {'keyword trap': 6, 'exception': 6, 'negation/constraint': 6, 'other (buried detail)': 6}
gold chunks per question: min=1 max=3; distinct gold chunks: 34
  A1 [keyword trap] gold: nt-063(p23, 194t)
  A2 [keyword trap] gold: nt-235(p59, 21t)
  A3 [keyword trap] gold: nt-185(p53, 71t)
  A4 [keyword trap] gold: nt-040(p19, 179t)
  A5 [keyword trap] gold: nt-043(p19, 161t), nt-044(p19, 149t)
  A6 [keyword trap] gold: nt-035(p18, 184t), nt-281(p63, 62t)
  E1 [exception   ] gold: nt-053(p22, 173t), nt-054(p22, 113t), nt-153(p44, 184t)
  E2 [exception   ] gold: nt-146(p42, 196t), nt-147(p42, 157t)
  E3 [exception   ] gold: nt-286(p69, 208t)
  E4 [exception   ] gold: nt-163(p48, 123t), nt-164(p48, 126t)
  E5 [exception   ] gold: nt-105(p34, 168t)
  E6 [exception   ] gold: nt-076(p27, 182t), nt-077(p27, 93t)
  N1 [negation/con] gold: nt-126(p37, 186t)
  N2 [negation/con] gold: nt-002(p9, 195t)
  N3 [negation/con] gold: nt-131(p38, 176t)
  N4 [negation/con] gold: nt-1

10611

---
# Stage N4. Frozen scoring rules, embeddings and baseline retrieval (Pipeline 1)
**What is frozen before this stage runs** (hashes are pinned in code; changing anything would stop the notebook):
* the corpus (305 chunks), the 24 preregistered questions and their gold labels, and the original preregistration;
* the scoring definitions: **overlay v1.1** (hash below), which supersedes v1.0 (kept unchanged for provenance);
* the models (`all-MiniLM-L6-v2` Bi-Encoder, `ms-marco-MiniLM-L-6-v2` Cross-Encoder) and all parameters (K = 10 -> 3, unit-length vectors, exact inner product).

**Reporting rules.** The preregistered **lenient** rule (any preregistered gold chunk in the Top 3) is the **primary** metric; the **strict**
rule of the overlay is a **secondary sensitivity analysis**. Both appear in the per-query table. Partial evidence is reported as partial and
never counted as strict success. Nothing (corpus, questions, labels, models, parameters) is changed after observing results.

In [12]:
OVERLAY_V1_1_SHA256 = "180c80dee8cf63207866cea103ec3c7c0e5dbeb80c9fd10d06b7e0460b8ab9ea"   # pinned 2026-10-18 16:06 IST, before any NIST embedding or retrieval
OVERLAY_V1_SHA256 = "c99f3cfe7cd7ec5898982fa499e9457dd2f2ed364a6a839c2424d7cd55d1a2ef"      # superseded by v1.1; kept for provenance
K_INIT, K_FINAL = 10, 3

overlay = json.load(open(OUT_DIR / "nist_scoring_overlay_v1_1.json", encoding="utf-8"))
core = {k: v for k, v in overlay.items() if k not in ("overlay_sha256", "status")}
recomputed = hashlib.sha256(json.dumps(core, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
v10_file = json.load(open(OUT_DIR / "nist_scoring_overlay_v1.json", encoding="utf-8"))
saved_pre = json.load(open(OUT_DIR / "nist_preregistration.json", encoding="utf-8"))
pre_recomputed = hashlib.sha256(json.dumps(dict(corpus_sha256=saved_pre["corpus_sha256"], queries=saved_pre["queries"]),
                                           sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
chain = {
    "overlay v1.1 hash recomputed == stored == pinned": recomputed == overlay["overlay_sha256"] == OVERLAY_V1_1_SHA256,
    "overlay v1.1 parents == frozen corpus and preregistration": (overlay["parents"]["corpus_sha256"] == FROZEN_CORPUS_SHA256
                                                                 and overlay["parents"]["preregistration_sha256"] == PREREG_SHA256),
    "overlay v1.1 supersedes the unchanged v1.0 file": v10_file["overlay_sha256"] == OVERLAY_V1_SHA256 == overlay["supersedes"]["overlay_sha256"]
                                                      and sha256_file(OUT_DIR / "nist_scoring_overlay_v1.json") == overlay["supersedes"]["file_sha256"],
    "preregistration FILE still hashes to the registered value": pre_recomputed == saved_pre["prereg_sha256"] == PREREG_SHA256,
    "in-memory questions == registered questions": [q["id"] for q in PREREG_Q] == [q["id"] for q in saved_pre["queries"]]
                                                   and [q["question"] for q in PREREG_Q] == [q["question"] for q in saved_pre["queries"]],
    "corpus file == pinned corpus hash": corpus_hash(load_frozen_chunks()) == FROZEN_CORPUS_SHA256,
}
for name, ok in chain.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(chain.values()), "the frozen evidence chain is broken; do not continue"

STRICT_RULES = {}
for q in PREREG_Q:
    ent = next((e for e in overlay["entries"] if e["question_id"] == q["id"]), None)
    if ent:
        r = ent["strict_rule"]
        STRICT_RULES[q["id"]] = dict(singles=[x["chunk"] for x in r["sufficient_single"]],
                                     joints=[[m["chunk"] for m in j["members"]] for j in r["joint_sets"]],
                                     excluded=[x["chunk"] for x in r["excluded_from_strict"]])
    else:
        STRICT_RULES[q["id"]] = dict(singles=list(q["gold_chunk_ids"]), joints=[], excluded=[])
print(f"strict rules loaded for {len(STRICT_RULES)} questions; {len(overlay['entries'])} differ from the preregistered gold set: {[e['question_id'] for e in overlay['entries']]}")

PASS  overlay v1.1 hash recomputed == stored == pinned
PASS  overlay v1.1 parents == frozen corpus and preregistration
PASS  overlay v1.1 supersedes the unchanged v1.0 file
PASS  preregistration FILE still hashes to the registered value
PASS  in-memory questions == registered questions
PASS  corpus file == pinned corpus hash
strict rules loaded for 24 questions; 8 differ from the preregistered gold set: ['A5', 'E1', 'E2', 'E4', 'N4', 'O2', 'O3', 'O5']


In [13]:
SELF_TESTS = []


def check(name, cond):
    assert cond, f"SELF-TEST FAILED: {name}"
    SELF_TESTS.append(name)


def rank_map(ranked):
    """1-based ranks of the candidate pool only: anything beyond the Top 10 is outside the pool."""
    return {c: i for i, c in enumerate(ranked[:K_INIT], 1)}


def score_lenient(gold, ranked):
    """PRIMARY: preregistered rule. success = any gold chunk in the Top 3; stumble = best gold at rank 4-10; miss = none in the Top 10."""
    pos = rank_map(ranked)
    ranks = sorted(pos[g] for g in gold if g in pos)
    best = ranks[0] if ranks else None
    outcome = "success" if best is not None and best <= K_FINAL else ("stumble" if best is not None else "miss")
    return dict(outcome=outcome, best_gold_rank=best, gold_ranks={g: pos.get(g) for g in gold})


def score_strict(rule, ranked):
    """SECONDARY: overlay v1.1. Only independently sufficient chunks, or complete joint sets, can give strict success."""
    pos, top = rank_map(ranked), set(ranked[:K_FINAL])
    singles_top = [s for s in rule["singles"] if s in top]
    joints_top = [j for j in rule["joints"] if all(m in top for m in j)]
    success = bool(singles_top or joints_top)
    partial_joint = [j for j in rule["joints"] if any(m in top for m in j) and not all(m in top for m in j)]
    excluded_top = [c for c in rule["excluded"] if c in top]
    top3_evidence = "sufficient" if success else ("partial" if (partial_joint or excluded_top) else "none")
    cands = [pos[s] for s in rule["singles"] if s in pos] + [max(pos[m] for m in j) for j in rule["joints"] if all(m in pos for m in j)]
    best = min(cands) if cands else None
    outcome = "success" if success else ("stumble" if best is not None else "miss")
    return dict(outcome=outcome, top3_evidence=top3_evidence, best_sufficient_rank=best,
                missing_joint_members=[[m for m in j if m not in top] for j in rule["joints"]])


def strict_role(qid, cid):
    r = STRICT_RULES[qid]
    if cid in r["singles"]: return "strict-single"
    for k, j in enumerate(r["joints"], 1):
        if cid in j: return f"joint-member-{k}"
    return "excluded-fragment" if cid in r["excluded"] else ""


filler = [f"z{i}" for i in range(20)]
rs = dict(singles=["a", "b"], joints=[], excluded=["x"])
rj = dict(singles=[], joints=[["p", "q"]], excluded=[])
check("strict: a single in the Top 3 is success", score_strict(rs, ["a"] + filler)["outcome"] == "success")
s = score_strict(rs, filler[:3] + ["b"] + filler[3:])
check("strict: a single at rank 4 is a stumble with no Top-3 evidence", s["outcome"] == "stumble" and s["top3_evidence"] == "none" and s["best_sufficient_rank"] == 4)
check("strict: a single outside the Top 10 is a miss", score_strict(rs, filler[:10] + ["a"])["outcome"] == "miss")
s = score_strict(rs, ["x"] + filler)
check("strict: an excluded fragment alone is PARTIAL and never success", s["outcome"] == "miss" and s["top3_evidence"] == "partial")
check("strict: fragment in the Top 3 plus a sufficient chunk at rank 5 is partial + stumble", score_strict(rs, ["x"] + filler[:3] + ["a"])["outcome"] == "stumble"
      and score_strict(rs, ["x"] + filler[:3] + ["a"])["top3_evidence"] == "partial")
check("strict: complete joint set in the Top 3 is success", score_strict(rj, ["p", "q"] + filler)["outcome"] == "success")
s = score_strict(rj, ["p"] + filler[:3] + ["q"])
check("strict: one joint member in the Top 3 and the other at rank 5 is PARTIAL + stumble, never success", s["outcome"] == "stumble" and s["top3_evidence"] == "partial" and s["best_sufficient_rank"] == 5)
s = score_strict(rj, ["p"] + filler)
check("strict: one joint member alone is partial + miss", s["outcome"] == "miss" and s["top3_evidence"] == "partial")
check("lenient: any gold chunk in the Top 3 is success", score_lenient(["x", "a"], ["x"] + filler)["outcome"] == "success")
check("lenient vs strict can diverge (fragment gives lenient success only)", score_lenient(["x"], ["x"] + filler)["outcome"] == "success" and score_strict(rs, ["x"] + filler)["outcome"] == "miss")
check("lenient: best gold at rank 4..10 is a stumble, outside is a miss",
      score_lenient(["a"], filler[:3] + ["a"] + filler[3:])["outcome"] == "stumble" and score_lenient(["a"], filler[:10] + ["a"])["outcome"] == "miss")
# the real overlay rules behave as specified
check("E1: nt-054 alone is partial, nt-053 alone is success", score_strict(STRICT_RULES["E1"], ["nt-054"] + filler)["top3_evidence"] == "partial"
      and score_strict(STRICT_RULES["E1"], ["nt-053"] + filler)["outcome"] == "success")
check("E2: nt-147 alone is partial", score_strict(STRICT_RULES["E2"], ["nt-147"] + filler)["top3_evidence"] == "partial")
check("N4: nt-162 alone is partial", score_strict(STRICT_RULES["N4"], ["nt-162"] + filler)["top3_evidence"] == "partial")
check("E4: both nt-163 and nt-164 are needed", score_strict(STRICT_RULES["E4"], ["nt-163", "nt-164"] + filler)["outcome"] == "success"
      and score_strict(STRICT_RULES["E4"], ["nt-163"] + filler)["outcome"] == "miss")
check("O3: both nt-121 and nt-122 are needed", score_strict(STRICT_RULES["O3"], ["nt-121", "nt-122"] + filler)["outcome"] == "success"
      and score_strict(STRICT_RULES["O3"], ["nt-122"] + filler)["top3_evidence"] == "partial")
check("O5: nt-301 alone is partial, with nt-300 it is success", score_strict(STRICT_RULES["O5"], ["nt-301"] + filler)["top3_evidence"] == "partial"
      and score_strict(STRICT_RULES["O5"], ["nt-300", "nt-301"] + filler)["outcome"] == "success")
check("A5: nt-043 alone is partial; nt-042 or nt-044 or nt-144 is success", score_strict(STRICT_RULES["A5"], ["nt-043"] + filler)["top3_evidence"] == "partial"
      and all(score_strict(STRICT_RULES["A5"], [c] + filler)["outcome"] == "success" for c in ("nt-042", "nt-044", "nt-144")))
check("A5: the removed joint set nt-145 + nt-146 gives no strict credit", score_strict(STRICT_RULES["A5"], ["nt-145", "nt-146"] + filler)["outcome"] == "miss")
check("every unchanged question: strict rule == preregistered gold set",
      all(STRICT_RULES[q] == dict(singles=list(next(x for x in PREREG_Q if x["id"] == q)["gold_chunk_ids"]), joints=[], excluded=[]) for q in overlay["unchanged_questions"]["ids"]))
ties = sorted(range(4), key=lambda i: (-[1.0, 2.0, 2.0, 0.5][i], i))
check("tie handling is deterministic: equal scores keep the original (baseline) order", ties == [1, 2, 0, 3])
print(f"{len(SELF_TESTS)} scoring self-tests passed")

21 scoring self-tests passed


## Stage N4a. Embed the frozen chunks and build the exact cosine index
Same model, normalisation and index type as the pilot. No query is embedded here.

In [14]:
import importlib.metadata as md
import platform

import faiss
import numpy as np
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer

INDEX_PATH, EMB_PATH = DATA_DIR / "index.faiss", DATA_DIR / "embeddings.npy"
CHUNKS = load_frozen_chunks()
assert corpus_hash(CHUNKS) == FROZEN_CORPUS_SHA256
texts = [c["text"] for c in CHUNKS]
torch.manual_seed(0)
model = SentenceTransformer(EMBED_MODEL, device="cpu")
DIM = getattr(model, "get_embedding_dimension", model.get_sentence_embedding_dimension)()
assert model.max_seq_length == MODEL_MAX_SEQ == 256
full_len = [len(model.tokenizer(t, truncation=False)["input_ids"]) for t in texts]
assert full_len == [c["n_tokens"] for c in CHUNKS] and max(full_len) <= model.max_seq_length
print(f"model {EMBED_MODEL}: dim={DIM}, max_seq_length={model.max_seq_length}; longest chunk {max(full_len)} tokens: no chunk is truncated ({len(texts)}/{len(texts)} fed in full)")

E = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
norms = np.linalg.norm(E, axis=1)
assert E.shape == (len(CHUNKS), DIM) and np.isfinite(E).all() and np.allclose(norms, 1.0, atol=1e-5)
print(f"embeddings {E.shape} float32, all finite; L2 norms min={norms.min():.6f} max={norms.max():.6f}")
index = faiss.IndexFlatIP(DIM)
index.add(E)
np.save(EMB_PATH, E)
faiss.write_index(index, str(INDEX_PATH))
assert index.ntotal == len(CHUNKS) and index.d == DIM and index.metric_type == faiss.METRIC_INNER_PRODUCT
D_self, I_self = index.search(E, 1)
self_ok = bool((I_self[:, 0] == np.arange(len(CHUNKS))).all() and D_self.min() > 0.9999)
E2 = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
det_ok = bool(np.allclose(E, E2, atol=1e-6))
assert self_ok and det_ok
INDEX_SHA256, EMB_SHA256 = sha256_file(INDEX_PATH), hashlib.sha256(E.tobytes()).hexdigest()
check("index: 305 vectors, d=384, inner product", index.ntotal == 305 and index.d == 384)
check("index: every chunk retrieves itself first (305/305)", self_ok)
check("embeddings: deterministic on re-encoding", det_ok)
print(f"FAISS IndexFlatIP ntotal={index.ntotal} d={index.d}; self-retrieval {len(CHUNKS)}/{len(CHUNKS)}; re-encode deterministic")
print(f"index file sha256 = {INDEX_SHA256}\nembeddings sha256 = {EMB_SHA256}")

W1018 17:09:23.223000 3353 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15162.62it/s]

model sentence-transformers/all-MiniLM-L6-v2: dim=384, max_seq_length=256; longest chunk 226 tokens: no chunk is truncated (305/305 fed in full)


embeddings (305, 384) float32, all finite; L2 norms min=1.000000 max=1.000000


FAISS IndexFlatIP ntotal=305 d=384; self-retrieval 305/305; re-encode deterministic
index file sha256 = 5ad15f18c9e17155191e814ceb8f1a3be0809fa80d2426553124c64491348021
embeddings sha256 = 9eb498f7e1576064ed02ecef4aa5398b5f15406d7aaf50a3fd026d53ab8c5652


## Stage N4b. Baseline retrieval: Top 10 for each of the 24 preregistered questions
The first 3 chunks of each ranked list form the baseline context. The complete Top 10 is persisted: **Stage N5 re-ranks exactly these candidates**.

In [15]:
def baseline_retrieve(query: str, k: int = K_INIT):
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
    scores, ids = index.search(q, k)
    return [dict(rank=r, chunk_id=CHUNKS[int(i)]["chunk_id"], score=float(s)) for r, (i, s) in enumerate(zip(ids[0], scores[0]), 1)]


BASE = []
for q in PREREG_Q:
    rows = baseline_retrieve(q["question"])
    assert [r["rank"] for r in rows] == list(range(1, K_INIT + 1)) and len({r["chunk_id"] for r in rows}) == K_INIT
    assert all(rows[i]["score"] >= rows[i + 1]["score"] for i in range(K_INIT - 1))
    ids = [r["chunk_id"] for r in rows]
    for r in rows:
        r["prereg_gold"] = r["chunk_id"] in q["gold_chunk_ids"]
        r["strict_role"] = strict_role(q["id"], r["chunk_id"])
    BASE.append(dict(id=q["id"], category=q["category"], question=q["question"], gold_chunk_ids=q["gold_chunk_ids"],
                     top10=rows, top3_chunk_ids=ids[:K_FINAL],
                     lenient=score_lenient(q["gold_chunk_ids"], ids), strict=score_strict(STRICT_RULES[q["id"]], ids)))

BASELINE_PATH = OUT_DIR / "nist_baseline_top10.json"
BASELINE_PATH.write_text(json.dumps(dict(
    stage="N4 baseline (Bi-Encoder only)", embed_model=EMBED_MODEL, k_init=K_INIT, k_final=K_FINAL, scores="cosine similarity (unit vectors, exact inner product)",
    corpus_sha256=FROZEN_CORPUS_SHA256, preregistration_sha256=PREREG_SHA256, overlay_v1_1_sha256=OVERLAY_V1_1_SHA256,
    index_file_sha256=INDEX_SHA256, embeddings_sha256=EMB_SHA256, queries=BASE), indent=1, ensure_ascii=False), encoding="utf-8")
with open(OUT_DIR / "nist_baseline_top10.csv", "w", encoding="utf-8") as f:
    f.write("query_id,rank,chunk_id,cosine_score,prereg_gold,strict_role\n")
    for b in BASE:
        for r in b["top10"]:
            f.write(f"{b['id']},{r['rank']},{r['chunk_id']},{r['score']:.6f},{int(r['prereg_gold'])},{r['strict_role']}\n")
BASELINE_SHA256 = sha256_file(BASELINE_PATH)
print(f"persisted results/nist_baseline_top10.json (+ .csv); sha256 = {BASELINE_SHA256}\n")

for b in BASE:
    print("=" * 110)
    print(f"{b['id']} [{b['category']}]  {b['question']}")
    print(f"   preregistered gold: {b['gold_chunk_ids']}")
    for r in b["top10"]:
        tags = [t for t in ("GOLD" if r["prereg_gold"] else "", r["strict_role"], "-> LLM context" if r["rank"] <= K_FINAL else "") if t]
        print(f"   {r['rank']:>2}  {r['chunk_id']}  cos={r['score']:.4f}  {' | '.join(tags)}")
    L, S_ = b["lenient"], b["strict"]
    print(f"   LENIENT (primary): {L['outcome'].upper()}  best gold rank {L['best_gold_rank']}   |   STRICT (secondary): {S_['outcome'].upper()}  Top-3 evidence: {S_['top3_evidence']}  best sufficient rank {S_['best_sufficient_rank']}")

persisted results/nist_baseline_top10.json (+ .csv); sha256 = c697c05d64304be40714750070aaba022f0540da347530d44f78d1a46f7c137d

A1 [keyword trap]  Which risk assessment orientation begins with the consequences we care about and our most critical assets, rather than with attackers or weaknesses?
   preregistered gold: ['nt-063']
    1  nt-064  cos=0.6846  -> LLM context
    2  nt-020  cos=0.6728  -> LLM context
    3  nt-063  cos=0.6664  GOLD | strict-single | -> LLM context
    4  nt-006  cos=0.6616  
    5  nt-117  cos=0.6589  
    6  nt-124  cos=0.6432  
    7  nt-238  cos=0.6425  
    8  nt-069  cos=0.6350  
    9  nt-129  cos=0.6331  
   10  nt-039  cos=0.6299  
   LENIENT (primary): SUCCESS  best gold rank 3   |   STRICT (secondary): SUCCESS  Top-3 evidence: sufficient  best sufficient rank 3
A2 [keyword trap]  Which term describes different experts arriving at the same results from the same data?
   preregistered gold: ['nt-235']
    1  nt-235  cos=0.7779  GOLD | strict-single | 

In [16]:
def tally(rows, key, sub):
    import collections
    return dict(collections.Counter(r[key][sub] for r in rows))


print("=" * 110)
print(f"BASELINE, N = {len(BASE)} preregistered questions")
print("   LENIENT (primary)  :", tally(BASE, "lenient", "outcome"))
print("   STRICT (secondary) :", tally(BASE, "strict", "outcome"), "| Top-3 evidence:", tally(BASE, "strict", "top3_evidence"))
for cat in dict.fromkeys(b["category"] for b in BASE):
    sub = [b for b in BASE if b["category"] == cat]
    print(f"   {cat:24s} lenient {tally(sub, 'lenient', 'outcome')}   strict {tally(sub, 'strict', 'outcome')}")
for kind in ("success", "stumble", "miss"):
    print(f"   lenient {kind:8s}: {[b['id'] for b in BASE if b['lenient']['outcome'] == kind]}")
    print(f"   strict  {kind:8s}: {[b['id'] for b in BASE if b['strict']['outcome'] == kind]}")
print("   strict partial in the Top 3 (never success):", [b["id"] for b in BASE if b["strict"]["top3_evidence"] == "partial"])
print("   lenient success but not strict success:", [b["id"] for b in BASE if b["lenient"]["outcome"] == "success" and b["strict"]["outcome"] != "success"])
print("   predicted distractor present in the baseline Top 3 (descriptive):",
      sum(bool(set(q["likely_distractors"]) & set(b["top3_chunk_ids"])) for q, b in zip(PREREG_Q, BASE)), "of", len(BASE))

BASELINE, N = 24 preregistered questions
   LENIENT (primary)  : {'success': 21, 'stumble': 3}
   STRICT (secondary) : {'success': 20, 'stumble': 4} | Top-3 evidence: {'sufficient': 20, 'none': 3, 'partial': 1}
   keyword trap             lenient {'success': 5, 'stumble': 1}   strict {'success': 5, 'stumble': 1}
   exception                lenient {'success': 5, 'stumble': 1}   strict {'success': 4, 'stumble': 2}
   negation/constraint      lenient {'success': 5, 'stumble': 1}   strict {'success': 5, 'stumble': 1}
   other (buried detail)    lenient {'success': 6}   strict {'success': 6}
   lenient success : ['A1', 'A2', 'A3', 'A5', 'A6', 'E1', 'E2', 'E4', 'E5', 'E6', 'N1', 'N2', 'N3', 'N4', 'N6', 'O1', 'O2', 'O3', 'O4', 'O5', 'O6']
   strict  success : ['A1', 'A2', 'A3', 'A5', 'A6', 'E1', 'E2', 'E5', 'E6', 'N1', 'N2', 'N3', 'N4', 'N6', 'O1', 'O2', 'O3', 'O4', 'O5', 'O6']
   lenient stumble : ['A4', 'E3', 'N5']
   strict  stumble : ['A4', 'E3', 'E4', 'N5']
   lenient miss    : []
   st

---
# Stage N5. Cross-Encoder re-ranking of exactly the persisted Top 10 (Pipeline 2)
* The candidates are **read back from `results/nist_baseline_top10.json`** (its hash is checked); nothing is re-retrieved, so the re-ranker can only
  reorder the baseline's ten chunks. Re-ranking is the only thing that differs from Pipeline 1.
* All ten (query, chunk) pairs are scored jointly by the Cross-Encoder. Scores are raw logits; **ties keep the baseline order** (deterministic).
* Cross-Encoder logits are **not comparable** with Bi-Encoder cosine similarities, so the tables below compare **rank positions and outcomes only**.
* The reranked Top 3 is scored with the same frozen lenient (primary) and strict (secondary) rules.

In [17]:
CE_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
ce = CrossEncoder(CE_NAME, device="cpu")
CE_MAX_LEN = ce.max_seq_length
CE_PARAMS_M = sum(p.numel() for p in ce.model.parameters()) / 1e6
assert CE_MAX_LEN == 512 and ce.num_labels == 1
print(f"{CE_NAME}: {CE_PARAMS_M:.1f}M parameters, max_seq_length={CE_MAX_LEN}, one output logit, default activation={ce.activation_fn}")

assert sha256_file(BASELINE_PATH) == BASELINE_SHA256, "the persisted baseline file changed after it was written"
saved_base = json.load(open(BASELINE_PATH, encoding="utf-8"))
assert saved_base["corpus_sha256"] == FROZEN_CORPUS_SHA256 and saved_base["overlay_v1_1_sha256"] == OVERLAY_V1_1_SHA256
assert [q["id"] for q in saved_base["queries"]] == [q["id"] for q in PREREG_Q]
pair_lens = [len(ce.tokenizer(b["question"], CHUNK_BY_ID[r["chunk_id"]]["text"], truncation=False)["input_ids"]) for b in saved_base["queries"] for r in b["top10"]]
assert max(pair_lens) <= CE_MAX_LEN
print(f"candidates loaded from disk: {len(saved_base['queries'])} queries x {K_INIT} chunks; {len(pair_lens)} (query, chunk) pairs, longest {max(pair_lens)} tokens (window {CE_MAX_LEN}): none truncated")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7938.89it/s]

cross-encoder/ms-marco-MiniLM-L-6-v2: 22.7M parameters, max_seq_length=512, one output logit, default activation=Identity()
candidates loaded from disk: 24 queries x 10 chunks; 240 (query, chunk) pairs, longest 245 tokens (window 512): none truncated


In [18]:
def rerank_saved(question, saved_rows):
    ids = [r["chunk_id"] for r in saved_rows]
    logits = ce.predict([(question, CHUNK_BY_ID[c]["text"]) for c in ids], batch_size=K_INIT, show_progress_bar=False, activation_fn=torch.nn.Identity())
    logits = [float(x) for x in logits]
    order = sorted(range(K_INIT), key=lambda i: (-logits[i], i))          # ties: keep the baseline order
    out = []
    for new_rank, i in enumerate(order, 1):
        orig = saved_rows[i]["rank"]
        out.append(dict(rank=new_rank, original_rank=orig, rank_shift=orig - new_rank, chunk_id=ids[i], ce_logit=logits[i]))
    assert sorted(r["chunk_id"] for r in out) == sorted(ids), "re-ranking changed the candidate pool"
    return out


def movement(base_outcome, new_outcome):
    if base_outcome == "miss": return "pool miss (re-ranker cannot help)"
    if base_outcome == "stumble": return "RESTORED into Top 3" if new_outcome == "success" else "not restored"
    return "kept in Top 3" if new_outcome == "success" else "DEMOTED out of Top 3"


RERANK = []
for q, b in zip(PREREG_Q, saved_base["queries"]):
    rr = rerank_saved(q["question"], b["top10"])
    for r in rr:
        r["prereg_gold"] = r["chunk_id"] in q["gold_chunk_ids"]
        r["strict_role"] = strict_role(q["id"], r["chunk_id"])
    ids = [r["chunk_id"] for r in rr]
    L, S_ = score_lenient(q["gold_chunk_ids"], ids), score_strict(STRICT_RULES[q["id"]], ids)
    RERANK.append(dict(id=q["id"], category=q["category"], question=q["question"], gold_chunk_ids=q["gold_chunk_ids"], baseline_top10=[r["chunk_id"] for r in b["top10"]],
                       reranked_top10=rr, reranked_top3_chunk_ids=ids[:K_FINAL], lenient=L, strict=S_,
                       lenient_movement=movement(b["lenient"]["outcome"], L["outcome"]), strict_movement=movement(b["strict"]["outcome"], S_["outcome"])))

RERANK_PATH = OUT_DIR / "nist_rerank_top10.json"
RERANK_PATH.write_text(json.dumps(dict(
    stage="N5 re-ranked (Bi-Encoder Top 10 -> Cross-Encoder -> Top 3)", cross_encoder=CE_NAME, ce_max_seq_length=CE_MAX_LEN,
    scores="Cross-Encoder raw logits (Identity activation); NOT comparable with Bi-Encoder cosines; ties keep the baseline order",
    baseline_file_sha256=BASELINE_SHA256, corpus_sha256=FROZEN_CORPUS_SHA256, preregistration_sha256=PREREG_SHA256,
    overlay_v1_1_sha256=OVERLAY_V1_1_SHA256, k_init=K_INIT, k_final=K_FINAL, queries=RERANK), indent=1, ensure_ascii=False), encoding="utf-8")
with open(OUT_DIR / "nist_rerank_top10.csv", "w", encoding="utf-8") as f:
    f.write("query_id,reranked_rank,original_rank,rank_shift,chunk_id,ce_logit,prereg_gold,strict_role\n")
    for r_ in RERANK:
        for r in r_["reranked_top10"]:
            f.write(f"{r_['id']},{r['rank']},{r['original_rank']},{r['rank_shift']},{r['chunk_id']},{r['ce_logit']:.6f},{int(r['prereg_gold'])},{r['strict_role']}\n")
RERANK_SHA256 = sha256_file(RERANK_PATH)

# tests of the Stage-5 contract
check("re-ranking: for every query the reranked set == the persisted baseline set (same ten chunks)",
      all(sorted(r_["baseline_top10"]) == sorted(x["chunk_id"] for x in r_["reranked_top10"]) for r_ in RERANK))
check("re-ranking: candidates were read from the persisted file, not re-retrieved", sha256_file(BASELINE_PATH) == BASELINE_SHA256)
check("re-ranking: ranks are 1..10 and rank_shift = original_rank - reranked_rank", all([x["rank"] for x in r_["reranked_top10"]] == list(range(1, 11))
      and all(x["rank_shift"] == x["original_rank"] - x["rank"] for x in r_["reranked_top10"]) for r_ in RERANK))
check("re-ranking: logits are non-increasing down the reranked list", all(all(r_["reranked_top10"][i]["ce_logit"] >= r_["reranked_top10"][i + 1]["ce_logit"] for i in range(9)) for r_ in RERANK))
print(f"persisted results/nist_rerank_top10.json (+ .csv); sha256 = {RERANK_SHA256}\n")

for r_ in RERANK:
    print("=" * 110)
    print(f"{r_['id']} [{r_['category']}]  gold: {r_['gold_chunk_ids']}")
    print(f"   {'new':>3} {'was':>3} {'shift':>5}  chunk   CE logit   flags")
    for x in r_["reranked_top10"]:
        tags = [t for t in ("GOLD" if x["prereg_gold"] else "", x["strict_role"], "-> LLM context" if x["rank"] <= K_FINAL else "") if t]
        print(f"   {x['rank']:>3} {x['original_rank']:>3} {x['rank_shift']:>+5}  {x['chunk_id']}  {x['ce_logit']:9.3f}   {' | '.join(tags)}")
    b = BASE[[q['id'] for q in PREREG_Q].index(r_["id"])]
    print(f"   LENIENT: {b['lenient']['outcome']} (gold rank {b['lenient']['best_gold_rank']}) -> {r_['lenient']['outcome']} (gold rank {r_['lenient']['best_gold_rank']})  [{r_['lenient_movement']}]")
    print(f"   STRICT : {b['strict']['outcome']} (evidence {b['strict']['top3_evidence']}, best sufficient rank {b['strict']['best_sufficient_rank']}) -> {r_['strict']['outcome']} (evidence {r_['strict']['top3_evidence']}, best sufficient rank {r_['strict']['best_sufficient_rank']})  [{r_['strict_movement']}]")

persisted results/nist_rerank_top10.json (+ .csv); sha256 = 1282c5190017579a26925ce6ad706c486b969eee997c590caeb786749c800838

A1 [keyword trap]  gold: ['nt-063']
   new was shift  chunk   CE logit   flags
     1   1    +0  nt-064      2.503   -> LLM context
     2   3    +1  nt-063      2.125   GOLD | strict-single | -> LLM context
     3   7    +4  nt-238     -3.443   -> LLM context
     4   5    +1  nt-117     -4.015   
     5   4    -1  nt-006     -4.220   
     6   2    -4  nt-020     -5.110   
     7   6    -1  nt-124     -5.297   
     8   9    +1  nt-129     -5.703   
     9   8    -1  nt-069     -6.748   
    10  10    +0  nt-039     -8.623   
   LENIENT: success (gold rank 3) -> success (gold rank 2)  [kept in Top 3]
   STRICT : success (evidence sufficient, best sufficient rank 3) -> success (evidence sufficient, best sufficient rank 2)  [kept in Top 3]
A2 [keyword trap]  gold: ['nt-235']
   new was shift  chunk   CE logit   flags
     1   1    +0  nt-235      6.556   GOLD | 

## Stage N5b. Per-query comparison, aggregates and the audit of genuine failure cases
A **genuine failure** = necessary evidence ranked 4-10 in the baseline and therefore excluded from the baseline Top 3 (under the lenient rule: no gold chunk in the Top 3 but one at rank 4-10;
under the strict rule: sufficient evidence, a single chunk or a complete joint set, within the Top 10 but not in the Top 3). A failure is **restored** if the reranked Top 3 then contains sufficient evidence.
Candidate-pool misses (nothing sufficient in the Top 10) cannot be restored by a re-ranker that only reorders the Top 10.

In [19]:
import collections
BY = {b["id"]: b for b in BASE}
print(f"{'id':3} {'category':22} | LENIENT base -> rerank (gold rank)            | STRICT base -> rerank (Top-3 evidence)")
for r_ in RERANK:
    b = BY[r_["id"]]
    l0, l1, s0, s1 = b["lenient"], r_["lenient"], b["strict"], r_["strict"]
    print(f"{r_['id']:3} {r_['category'][:22]:22} | {l0['outcome']:7s}({str(l0['best_gold_rank']):>4}) -> {l1['outcome']:7s}({str(l1['best_gold_rank']):>4}) {r_['lenient_movement']:34s} | "
          f"{s0['outcome']:7s}[{s0['top3_evidence'][:4]}] -> {s1['outcome']:7s}[{s1['top3_evidence'][:4]}] {r_['strict_movement']}")

with open(OUT_DIR / "nist_per_query_results.csv", "w", encoding="utf-8") as f:
    f.write("query_id,category,lenient_baseline,lenient_baseline_best_gold_rank,lenient_reranked,lenient_reranked_best_gold_rank,lenient_movement,"
            "strict_baseline,strict_baseline_top3_evidence,strict_baseline_best_sufficient_rank,strict_reranked,strict_reranked_top3_evidence,strict_reranked_best_sufficient_rank,strict_movement\n")
    for r_ in RERANK:
        b = BY[r_["id"]]
        f.write(",".join(str(x) for x in (r_["id"], r_["category"], b["lenient"]["outcome"], b["lenient"]["best_gold_rank"], r_["lenient"]["outcome"], r_["lenient"]["best_gold_rank"],
                                          r_["lenient_movement"].replace(",", ";"), b["strict"]["outcome"], b["strict"]["top3_evidence"], b["strict"]["best_sufficient_rank"],
                                          r_["strict"]["outcome"], r_["strict"]["top3_evidence"], r_["strict"]["best_sufficient_rank"], r_["strict_movement"].replace(",", ";"))) + "\n")

print("\n" + "=" * 110)
for label, key, mkey in (("LENIENT (primary)", "lenient", "lenient_movement"), ("STRICT (secondary)", "strict", "strict_movement")):
    print(f"{label}: baseline {dict(collections.Counter(b[key]['outcome'] for b in BASE))} -> reranked {dict(collections.Counter(r_[key]['outcome'] for r_ in RERANK))}")
    print(f"   movements: {dict(collections.Counter(r_[mkey] for r_ in RERANK))}")

def failures(key):
    return [r_ for r_ in RERANK if BY[r_["id"]][key]["outcome"] == "stumble"]


audit = {}
for label, key, mkey in (("LENIENT (primary)", "lenient", "lenient_movement"), ("STRICT (secondary)", "strict", "strict_movement")):
    fl = failures(key)
    restored = [r_ for r_ in fl if r_[mkey].startswith("RESTORED")]
    demoted = [r_ for r_ in RERANK if r_[mkey].startswith("DEMOTED")]
    audit[key] = dict(failures=[r_["id"] for r_ in fl], restored=[r_["id"] for r_ in restored], demoted=[r_["id"] for r_ in demoted],
                      pool_misses=[r_["id"] for r_ in RERANK if BY[r_["id"]][key]["outcome"] == "miss"])
    print(f"\n--- failure audit, {label} ---")
    print(f"genuine baseline failures (necessary evidence at rank 4-10, excluded from the Top 3): {len(fl)} -> {audit[key]['failures']}")
    for r_ in fl:
        b = BY[r_["id"]]
        if key == "lenient":
            before, after = b["lenient"]["gold_ranks"], r_["lenient"]["gold_ranks"]
        else:
            rule = STRICT_RULES[r_["id"]]
            evidence = rule["singles"] + [m for j in rule["joints"] for m in j]
            bpos = {x["chunk_id"]: x["rank"] for x in b["top10"]}
            rpos = {x["chunk_id"]: x["rank"] for x in r_["reranked_top10"]}
            before, after = {c: bpos.get(c) for c in evidence}, {c: rpos.get(c) for c in evidence}
        print(f"   {r_['id']}: evidence chunk ranks baseline {before} -> reranked {after}  => {r_[mkey]}")
    print(f"RESTORED failures: {len(restored)} of {len(fl)} -> {audit[key]['restored']}")
    print(f"DEMOTIONS (baseline success -> not success): {len(demoted)} -> {audit[key]['demoted']}")
    print(f"candidate-pool misses (cannot be restored by reordering the Top 10): {len(audit[key]['pool_misses'])} -> {audit[key]['pool_misses']}")
    print(f"assignment requirement (>= 2 restored failures): {'MET' if len(restored) >= 2 else 'NOT MET'} ({len(restored)})")

lenient_restored = len(audit["lenient"]["restored"])
if lenient_restored < 2:
    print("\nSHORTFALL: fewer than two failures were restored under the primary (lenient) rule. This result is reported as observed. Nothing (queries, labels, chunking, models, parameters) was changed to alter it.")
    print("Proposed remedy, NOT executed (needs your review and a separate, dated, hashed preregistration):")
    print("  1. Write a second question batch, with the same four categories, from the frozen corpus, BEFORE embedding or running any of its questions; verify its gold labels by string match and a read-through; hash it.")
    print("  2. Run it through the same frozen pipelines (same models, K, scoring definitions) and report it as a separate batch (N = 24 + the new N), never silently merged with batch 1.")
    print("  3. State in the report how many questions were tried and how many produced restored failures.")
    print("  The honest alternative is to report the shortfall itself as the finding.")

# descriptive only: gold rank movement of the best gold chunk
moved = [(r_["id"], BY[r_["id"]]["lenient"]["best_gold_rank"], r_["lenient"]["best_gold_rank"]) for r_ in RERANK if BY[r_["id"]]["lenient"]["best_gold_rank"] and BY[r_["id"]]["lenient"]["best_gold_rank"] != r_["lenient"]["best_gold_rank"]]
print(f"\nbest preregistered-gold rank changed for {len(moved)} queries (id, before, after): {moved}")
changed3 = [r_["id"] for r_ in RERANK if set(BY[r_["id"]]["top3_chunk_ids"]) != set(r_["reranked_top3_chunk_ids"])]
print(f"Top-3 membership changed for {len(changed3)} of {len(RERANK)} queries: {changed3}")
check("audit: restored failures are a subset of failures, and counts are consistent", all(set(audit[k]["restored"]) <= set(audit[k]["failures"]) for k in audit))

id  category               | LENIENT base -> rerank (gold rank)            | STRICT base -> rerank (Top-3 evidence)
A1  keyword trap           | success(   3) -> success(   2) kept in Top 3                      | success[suff] -> success[suff] kept in Top 3
A2  keyword trap           | success(   1) -> success(   1) kept in Top 3                      | success[suff] -> success[suff] kept in Top 3
A3  keyword trap           | success(   1) -> success(   1) kept in Top 3                      | success[suff] -> success[suff] kept in Top 3
A4  keyword trap           | stumble(   6) -> success(   1) RESTORED into Top 3                | stumble[none] -> success[suff] RESTORED into Top 3
A5  keyword trap           | success(   2) -> success(   1) kept in Top 3                      | success[suff] -> success[suff] kept in Top 3
A6  keyword trap           | success(   1) -> success(   1) kept in Top 3                      | success[suff] -> success[suff] kept in Top 3
E1  exception             

---
# Stage N6. Answer generation for both pipelines (Groq), and the answer comparison
**The retrieval evaluation (Stages 4-5) is frozen.** Stage 6 reads the persisted Top-3 contexts, verifies them against pinned hashes, and sends them to **one** LLM with **one** prompt and **one** set of parameters:
Answer 1 = the baseline Top 3 (Bi-Encoder), Answer 2 = the re-ranked Top 3 (Cross-Encoder). **The selected chunks, and their order, are the only intended difference.**

* Nothing in this section can change the 24 questions, the gold evidence, the chunks, the Top-10 candidates, the re-ranking, the scoring rules or the retrieval models.
* The network is used **only** when `RAG_STAGE6_MODE` is `smoke` or `evaluate`; the default is `off`, so *Run All* never calls an API.
* Answer quality is judged separately from retrieval success (a retrieved gold chunk does not guarantee a correct answer), and correctness is a human judgement on a review sheet, never an LLM judge.

## Stage N6a. The fixed prompt (identical for both pipelines)
The instruction, the context format and the generation parameters below were fixed **before any provider was chosen and before any answer existed**.
Answer 1 uses the baseline Top 3 and Answer 2 uses the re-ranked Top 3; **the selected Top-3 chunks are the only intended difference**. No LLM is called in this cell.

In [2]:
SYSTEM_INSTRUCTION = (
    "You answer questions about NIST Special Publication 800-30 Revision 1 using ONLY the context passages provided. "
    "If the context does not contain the answer, reply exactly: The provided context does not contain the answer. "
    "Do not use outside knowledge. Answer in at most three sentences and cite the passage ids you used in square brackets, for example [nt-012].")
CONTEXT_HEADER = "Context passages (in the order retrieved):"
PROMPT_TEMPLATE = "{system}\n\n{header}\n{context}\n\nQuestion: {question}\nAnswer:"
GENERATION_CONFIG = dict(temperature=0.0, top_p=1.0, max_output_tokens=300, n=1)      # identical for both pipelines; provider-specific names are mapped in N6b


def format_context(chunk_ids, chunk_by_id=None):
    """Same format for both pipelines; passages appear in the pipeline's own ranking order."""
    by_id = CHUNK_BY_ID if chunk_by_id is None else chunk_by_id
    return "\n\n".join(f"[{cid}] (PDF page {by_id[cid]['page_start']})\n{by_id[cid]['text']}" for cid in chunk_ids)


def build_prompt(question, chunk_ids, chunk_by_id=None):
    return PROMPT_TEMPLATE.format(system=SYSTEM_INSTRUCTION, header=CONTEXT_HEADER, context=format_context(chunk_ids, chunk_by_id), question=question)


def build_messages(question, chunk_ids, chunk_by_id=None):
    """The same prompt as build_prompt(), split at the blank line after the instruction: system = the instruction, user = context + question."""
    user = f"{CONTEXT_HEADER}\n{format_context(chunk_ids, chunk_by_id)}\n\nQuestion: {question}\nAnswer:"
    return [dict(role="system", content=SYSTEM_INSTRUCTION), dict(role="user", content=user)]


# tests: string building only (no network, no LLM)
split = lambda p: (p.split(CONTEXT_HEADER)[0], p.split("\n\nQuestion: ")[1])
for b_, r_ in zip(BASE, RERANK):
    assert b_["id"] == r_["id"] and b_["question"] == r_["question"]
    pa, pb = build_prompt(b_["question"], b_["top3_chunk_ids"]), build_prompt(b_["question"], r_["reranked_top3_chunk_ids"])
    assert build_prompt(b_["question"], b_["top3_chunk_ids"]) == pa
    assert split(pa) == split(pb), f"{b_['id']}: the prompts differ outside the context block"
    assert all(CHUNK_BY_ID[c]["text"] in pa for c in b_["top3_chunk_ids"]) and all(CHUNK_BY_ID[c]["text"] in pb for c in r_["reranked_top3_chunk_ids"])
    unselected = [c for c in CHUNK_BY_ID if c not in b_["top3_chunk_ids"] and c not in r_["reranked_top3_chunk_ids"]]
    assert all(CHUNK_BY_ID[c]["text"] not in pa and CHUNK_BY_ID[c]["text"] not in pb for c in unselected[:25])
    assert "\n\n".join(m["content"] for m in build_messages(b_["question"], b_["top3_chunk_ids"])) == pa
    assert "\n\n".join(m["content"] for m in build_messages(b_["question"], r_["reranked_top3_chunk_ids"])) == pb
check("prompt: for all 24 queries, building twice from the same inputs is identical and everything outside the context block is identical for the two pipelines", True)
check("prompt: for all 24 queries, each prompt holds exactly its pipeline's selected chunks and no unselected chunk", True)
check("prompt: the system + user messages join back to exactly the template string (nothing added or lost by the split)", True)
check("prompt: Answer 1 and Answer 2 contexts differ for all 24 queries (order or membership), as the Stage 5 audit found",
      sum(b_["top3_chunk_ids"] != r_["reranked_top3_chunk_ids"] for b_, r_ in zip(BASE, RERANK)) == 24
      and sum(set(b_["top3_chunk_ids"]) != set(r_["reranked_top3_chunk_ids"]) for b_, r_ in zip(BASE, RERANK)) == 22)
PROMPT_SHA256 = hashlib.sha256(json.dumps(dict(system=SYSTEM_INSTRUCTION, header=CONTEXT_HEADER, template=PROMPT_TEMPLATE, config=GENERATION_CONFIG), sort_keys=True).encode()).hexdigest()
assert PROMPT_SHA256 == "013f4a4cbaf737a3b498ebfd23d06c1293d55c37024c9baab09e82da9a37300c", "the fixed prompt or the generation parameters changed since they were frozen"
print(f"prompt + generation parameters sha256 = {PROMPT_SHA256} (unchanged since it was frozen in Stage 5); no LLM was called")
print("\n--- the fixed prompt, shown with the first question and its baseline context truncated ---")
print(build_prompt(BASE[0]["question"], BASE[0]["top3_chunk_ids"])[:700] + " ...")

prompt + generation parameters sha256 = 013f4a4cbaf737a3b498ebfd23d06c1293d55c37024c9baab09e82da9a37300c (unchanged since it was frozen in Stage 5); no LLM was called

--- the fixed prompt, shown with the first question and its baseline context truncated ---
You answer questions about NIST Special Publication 800-30 Revision 1 using ONLY the context passages provided. If the context does not contain the answer, reply exactly: The provided context does not contain the answer. Do not use outside knowledge. Answer in at most three sentences and cite the passage ids you used in square brackets, for example [nt-012].

Context passages (in the order retrieved):
[nt-064] (PDF page 24)
A vulnerability-oriented approach starts with a set of predisposing conditions or exploitable weaknesses/deficiencies in organizational information systems or the environments in which the systems operate, and identifies threat events that could exercise those vulnerabili ...


## Stage N6b. Provider (Groq), credentials, request mapping and pacing
**Mode switch.** `RAG_STAGE6_MODE` decides whether this notebook may touch the network: `off` (the default: no network access at all), `smoke` (two synthetic non-evaluation prompts), `evaluate` (the 48 evaluation answers; refuses to start unless a smoke test passed for exactly these settings).
**Credentials.** The key is read only from the environment variable `GROQ_API_KEY`, optionally loaded from a git-ignored `.env`. It is never printed, logged, or written to any result file; error text is redacted before it is stored.
**Provider facts** below were read from Groq's official documentation on 2026-10-18 and are *not* guaranteed: the account's own *Limits* page is authoritative, and every response's rate-limit headers are logged.

In [3]:
import datetime
import subprocess
import time
import urllib.error
from collections import deque

STAGE6_MODE = os.environ.get("RAG_STAGE6_MODE", "off")      # off (default) | smoke | evaluate
assert STAGE6_MODE in ("off", "smoke", "evaluate"), f"RAG_STAGE6_MODE must be off, smoke or evaluate (got {STAGE6_MODE!r})"

GEN_PROVIDER = "groq"
GEN_BASE_URL = "https://api.groq.com/openai/v1"
GEN_MODEL = "qwen/qwen3.8-27b"          # the recommended free-tier model (see README / report); changing it changes SETTINGS_SHA256 and the resume guard refuses mixed ledgers
GEN_SEED = 1234                         # Groq: best-effort determinism only; identical for both pipelines
GEN_EXTRAS_BY_MODEL = {"qwen/qwen3.8-27b": dict(reasoning_effort="none")}    # "none" = no reasoning tokens, so the 300-token cap applies to the visible answer
GEN_TIMEOUT_S = 90.0
MAX_ATTEMPTS, NO_CONTENT_MAX_ATTEMPTS, BACKOFF_CAP_S = 6, 2, 60.0
TOKENS_PER_CHAR = 1 / 3.0               # deliberately conservative input-size estimate (typical English is about 1/4); only used for pacing

GROQ_DOCS_SNAPSHOT = dict(
    read_on="2026-10-18",
    urls=["https://console.groq.com/docs/rate-limits", "https://console.groq.com/docs/models", "https://console.groq.com/docs/reasoning",
          "https://console.groq.com/docs/deprecations", "https://console.groq.com/docs/your-data", "https://console.groq.com/docs/errors"],
    free_plan_limits={m: dict(rpm=30, rpd=1000, tpm=8000, tpd=200000) for m in ("openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b")},
    status={"openai/gpt-oss-120b": "production", "openai/gpt-oss-20b": "production", "qwen/qwen3.8-27b": "preview (may be discontinued at short notice)"},
    notes=["Limits apply per organisation; whichever limit is hit first applies; the documented figures are a summary and the account's Limits page is authoritative.",
           "Free-plan chat models listed: gpt-oss-120b, gpt-oss-20b, gpt-oss-safeguard-20b (safety classifier), qwen3.8-27b. Llama 3.x chat models are marked Enterprise / Contact Sales and were deprecated for free and developer tiers on 2026-08-16.",
           "A temperature of 0 is converted by Groq to 1e-8; n must be 1; seed is best-effort only; reasoning tokens of the gpt-oss models are part of the completion tokens and cannot be switched off.",
           "Inference inputs and outputs are not retained by default; they may be logged for up to 30 days for reliability or abuse investigation unless Zero Data Retention is enabled."])
TPM_LIMIT = GROQ_DOCS_SNAPSHOT["free_plan_limits"][GEN_MODEL]["tpm"]
TPM_BUDGET = int(0.80 * TPM_LIMIT)       # stay clearly under the documented per-minute token limit


def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()


def provider_request(model, messages):
    """Map the provider-neutral GENERATION_CONFIG to Groq's field names. The only provider-specific additions are the seed and reasoning_effort."""
    cfg = GENERATION_CONFIG
    body = dict(model=model, messages=messages, temperature=cfg["temperature"], top_p=cfg["top_p"], max_completion_tokens=cfg["max_output_tokens"],
                n=cfg["n"], seed=GEN_SEED, stream=False)
    body.update(GEN_EXTRAS_BY_MODEL[model])
    return body


REQUEST_TEMPLATE = {k: v for k, v in provider_request(GEN_MODEL, []).items() if k != "messages"}
SETTINGS_SHA256 = sha256_json(dict(provider=GEN_PROVIDER, base_url=GEN_BASE_URL, request_template=REQUEST_TEMPLATE,
                                   message_layout="system = SYSTEM_INSTRUCTION; user = context header + passages + question", prompt_and_generation_config_sha256=PROMPT_SHA256))
print(f"provider {GEN_PROVIDER}, model {GEN_MODEL}; mode = {STAGE6_MODE}")
print(f"request settings (everything except the messages): {REQUEST_TEMPLATE}")
print(f"settings sha256 = {SETTINGS_SHA256}")


# ---- credentials ---------------------------------------------------------------------------------------------------------------
ENV_FILE = Path(".env")


def load_env_file(path=ENV_FILE, environ=None, is_git_ignored=None):
    """Minimal KEY=VALUE reader for the git-ignored .env. Refuses to read the file unless git ignores it; never overrides variables that
    are already set; returns only the NAMES it set, never a value."""
    environ = os.environ if environ is None else environ
    if not Path(path).exists():
        return []
    if is_git_ignored is None:
        is_git_ignored = lambda p: subprocess.run(["git", "check-ignore", "-q", str(p)], capture_output=True).returncode == 0
    if not is_git_ignored(path):
        raise RuntimeError(f"{path} is NOT ignored by git: refusing to load credentials from it. Add it to .gitignore, or export the variable in your shell.")
    loaded = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name, value = name.strip().removeprefix("export ").strip(), value.strip().strip('"').strip("'")
        if name and value and name not in environ:
            environ[name] = value
            loaded.append(name)
    return loaded


def redact(text, secret):
    return text.replace(secret, "[REDACTED]") if secret else text


SAFE_HEADERS = ("retry-after", "x-ratelimit-limit-requests", "x-ratelimit-remaining-requests", "x-ratelimit-limit-tokens", "x-ratelimit-remaining-tokens",
                "x-ratelimit-reset-requests", "x-ratelimit-reset-tokens", "x-request-id", "date")


def safe_headers(headers):
    low = {str(k).lower(): v for k, v in (headers or {}).items()}
    return {k: low[k] for k in SAFE_HEADERS if k in low}


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def make_groq_transport(api_key, base_url=GEN_BASE_URL, timeout_s=GEN_TIMEOUT_S):
    """Standard-library HTTPS client. The key lives only in this closure; nothing here prints or stores it."""
    def post(body):
        req = urllib.request.Request(f"{base_url}/chat/completions", data=json.dumps(body).encode("utf-8"), method="POST",
                                     headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", "User-Agent": "rag-assignment-nist/1.0"})
        t0 = time.perf_counter()
        try:
            with urllib.request.urlopen(req, timeout=timeout_s) as resp:
                raw, status, headers = resp.read(), resp.status, dict(resp.headers.items())
        except urllib.error.HTTPError as e:
            raw, status, headers = e.read(), e.code, dict(e.headers.items())
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            return dict(status=None, headers={}, body=None, error=redact(f"{type(e).__name__}: {e}", api_key)[:300], elapsed_s=time.perf_counter() - t0)
        elapsed = time.perf_counter() - t0
        try:
            parsed = json.loads(raw.decode("utf-8"))
        except ValueError:
            parsed = None
        return dict(status=status, headers=headers, body=parsed, error=None if parsed is not None else "non-JSON response body", elapsed_s=elapsed)
    return post


def make_groq_model_lister(api_key, base_url=GEN_BASE_URL, timeout_s=30.0):
    def list_models():
        req = urllib.request.Request(f"{base_url}/models", method="GET", headers={"Authorization": f"Bearer {api_key}", "User-Agent": "rag-assignment-nist/1.0"})
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            return sorted(m["id"] for m in json.loads(resp.read().decode("utf-8"))["data"])
    return list_models


class TokenWindow:
    """Sliding 60-second window over estimated tokens (input estimate + the full max output), so requests are paced under the per-minute limit."""

    def __init__(self, budget, clock=time.monotonic, sleep=time.sleep):
        self.budget, self.clock, self.sleep, self.events = budget, clock, sleep, deque()

    def wait(self, est_tokens):
        waited = 0.0
        while True:
            now = self.clock()
            while self.events and now - self.events[0][0] >= 60.0:
                self.events.popleft()
            if not self.events or sum(t for _, t in self.events) + est_tokens <= self.budget:
                return waited
            delay = self.events[0][0] + 60.0 - now + 0.05
            self.sleep(delay)
            waited += delay

    def record(self, est_tokens):
        self.events.append((self.clock(), est_tokens))


def estimate_tokens(messages):
    return int(sum(len(m["content"]) for m in messages) * TOKENS_PER_CHAR) + 1

provider groq, model qwen/qwen3.8-27b; mode = off
request settings (everything except the messages): {'model': 'qwen/qwen3.8-27b', 'temperature': 0.0, 'top_p': 1.0, 'max_completion_tokens': 300, 'n': 1, 'seed': 1234, 'stream': False, 'reasoning_effort': 'none'}
settings sha256 = fc869685494b2d61c1bc68fc5888a7fb21ac31eaf868c41fcb6a7e60262d967c


## Stage N6c. The 48 generation tasks, the resumable generator, and the audit trail
* **Inputs are the persisted Stage 4/5 files, verified against pinned hashes** (a mismatch stops the notebook; nothing is regenerated to make it pass).
* **Tasks**: for each of the 24 queries, Answer 1 (baseline Top 3) and Answer 2 (re-ranked Top 3), same model, same messages layout, same parameters. The call order inside a pair alternates by query so neither pipeline always goes first (relevant for latency).
* **Resumable**: every finished answer is appended to a ledger (`nist_stage6_answers.jsonl`); every API attempt, successful or not, is appended to `nist_stage6_attempts.jsonl`. A re-run skips answers already on disk and refuses a ledger written under different settings or prompts.
* **Retries** (timeouts, network errors, 429, 5xx, empty content) resend the byte-identical request; they never change the model or any parameter. Bad requests and authentication errors abort the run instead of being retried.

In [4]:
BASELINE_FILE_SHA256_PIN = "c697c05d64304be40714750070aaba022f0540da347530d44f78d1a46f7c137d"      # nist_baseline_top10.json as verified at the end of Stage 5
RERANK_FILE_SHA256_PIN = "1282c5190017579a26925ce6ad706c486b969eee997c590caeb786749c800838"        # nist_rerank_top10.json   as verified at the end of Stage 5
REFUSAL = "The provided context does not contain the answer."

S6_PROMPT_SPEC = OUT_DIR / "nist_stage6_prompt_spec.json"
S6_SMOKE = OUT_DIR / "nist_stage6_smoke_test.json"
S6_LEDGER = OUT_DIR / "nist_stage6_answers.jsonl"
S6_ATTEMPTS = OUT_DIR / "nist_stage6_attempts.jsonl"
S6_ANSWERS = OUT_DIR / "nist_stage6_answers.json"
S6_COMPARISON = OUT_DIR / "nist_stage6_comparison.csv"


class GenerationAbort(RuntimeError):
    """A condition that retrying cannot fix (bad request, authentication, model mismatch): the run stops and keeps everything already saved."""


def load_frozen_retrieval():
    for path, pin in ((BASELINE_PATH, BASELINE_FILE_SHA256_PIN), (RERANK_PATH, RERANK_FILE_SHA256_PIN)):
        if sha256_file(path) != pin:
            raise RuntimeError(f"{path.name} differs from its pinned hash: the frozen retrieval result changed. Stop and investigate; do not regenerate it.")
    base, rer = json.load(open(BASELINE_PATH, encoding="utf-8")), json.load(open(RERANK_PATH, encoding="utf-8"))
    assert base["corpus_sha256"] == rer["corpus_sha256"] == FROZEN_CORPUS_SHA256 and base["overlay_v1_1_sha256"] == rer["overlay_v1_1_sha256"] == OVERLAY_V1_1_SHA256
    assert rer["baseline_file_sha256"] == BASELINE_FILE_SHA256_PIN
    ids = [q["id"] for q in PREREG_Q]
    assert [q["id"] for q in base["queries"]] == [q["id"] for q in rer["queries"]] == ids
    assert all(b["question"] == r["question"] == q["question"] for b, r, q in zip(base["queries"], rer["queries"], PREREG_Q))
    return base["queries"], rer["queries"]


def build_tasks(base_q, rer_q):
    tasks = []
    for i, (b, r) in enumerate(zip(base_q, rer_q)):
        pair = [("baseline", "Answer 1", list(b["top3_chunk_ids"])), ("reranked", "Answer 2", list(r["reranked_top3_chunk_ids"]))]
        if i % 2 == 1:
            pair.reverse()                                                  # alternate which pipeline is called first
        for position, (pipeline, label, ids) in enumerate(pair, 1):
            msgs = build_messages(b["question"], ids)
            tasks.append(dict(key=f"{b['id']}|{pipeline}", query_id=b["id"], pipeline=pipeline, label=label, context_chunk_ids=ids, messages=msgs,
                              prompt_sha256=sha256_json(msgs), call_position_in_pair=position))
    return tasks


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def classify(res, expected_model):
    """-> (verdict, detail). verdicts: ok | retry | no_content | task_failed | fatal."""
    st = res["status"]
    if st is None:
        return "retry", res.get("error") or "network error"
    if st == 200:
        b = res["body"] or {}
        try:
            content = b["choices"][0]["message"].get("content")
        except (KeyError, IndexError, TypeError, AttributeError):
            return "retry", "malformed 200 response"
        if b.get("model") != expected_model:
            return "fatal", f"the API answered with model {b.get('model')!r}, expected {expected_model!r}: refusing to continue"
        if not (content or "").strip():
            return "no_content", f"empty answer (finish_reason={b['choices'][0].get('finish_reason')})"
        return "ok", ""
    message = ((res.get("body") or {}).get("error") or {}).get("message", "") if isinstance(res.get("body"), dict) else ""
    if st in (400, 401, 403, 404):
        return "fatal", f"HTTP {st}: {message[:200]}"
    if st in (408, 422, 429, 500, 502, 503, 504):
        return "retry", f"HTTP {st}"
    return "task_failed", f"HTTP {st}: {message[:200]}"


def retry_after_seconds(headers):
    try:
        return float({str(k).lower(): v for k, v in (headers or {}).items()}.get("retry-after", 0))
    except (TypeError, ValueError):
        return 0.0


def generate_one(task, transport, window, sleep, expected_model, attempts_path, now=utc_now):
    body = provider_request(GEN_MODEL, task["messages"])
    request_sha256 = sha256_json(body)
    est = estimate_tokens(task["messages"]) + GENERATION_CONFIG["max_output_tokens"]
    waited_total, verdict, detail = 0.0, "", ""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        waited_total += window.wait(est)
        window.record(est)
        started = now()
        res = transport(body)
        verdict, detail = classify(res, expected_model)
        append_jsonl(attempts_path, dict(key=task["key"], attempt=attempt, started_at_utc=started, elapsed_s=round(res["elapsed_s"], 4), http_status=res["status"],
                                         verdict=verdict, detail=detail[:300], request_sha256=request_sha256, headers=safe_headers(res.get("headers"))))
        if verdict == "ok":
            b = res["body"]
            choice = b["choices"][0]
            return dict(status="ok", key=task["key"], query_id=task["query_id"], pipeline=task["pipeline"], label=task["label"],
                        answer=choice["message"]["content"].strip(), finish_reason=choice.get("finish_reason"), truncated=choice.get("finish_reason") == "length",
                        reasoning_returned=bool(choice["message"].get("reasoning")), context_chunk_ids=task["context_chunk_ids"], prompt_sha256=task["prompt_sha256"],
                        request_sha256=request_sha256, settings_sha256=SETTINGS_SHA256, model_requested=GEN_MODEL, model_returned=b.get("model"),
                        system_fingerprint=b.get("system_fingerprint"), response_id=b.get("id"), usage=b.get("usage"), started_at_utc=started, finished_at_utc=now(),
                        latency_s=round(res["elapsed_s"], 4), attempts=attempt, paced_wait_s=round(waited_total, 2), call_position_in_pair=task["call_position_in_pair"])
        if verdict == "fatal":
            raise GenerationAbort(f"{task['key']}: {detail}")
        if verdict == "task_failed":
            break
        if attempt >= (NO_CONTENT_MAX_ATTEMPTS if verdict == "no_content" else MAX_ATTEMPTS):
            break
        delay = min(BACKOFF_CAP_S, 2.0 ** attempt)
        if res["status"] == 429:
            delay = max(delay, retry_after_seconds(res.get("headers")))
        sleep(delay)
        waited_total += delay
    return dict(status="failed", key=task["key"], query_id=task["query_id"], pipeline=task["pipeline"], label=task["label"], attempts=attempt, last_verdict=verdict, last_detail=detail[:300])


def load_ledger(path, tasks_by_key):
    """Read finished answers; refuse anything produced under other settings or prompts. A torn final line (killed mid-write) is dropped."""
    done = {}
    if not Path(path).exists():
        return done
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]
    records = []
    for n, ln in enumerate(lines):
        try:
            records.append(json.loads(ln))
        except ValueError:
            if n != len(lines) - 1:
                raise RuntimeError(f"{path}: line {n + 1} is corrupt and is not the last line: stop and inspect it")
            Path(path).write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records), encoding="utf-8")     # drop the torn tail
    for rec in records:
        t = tasks_by_key.get(rec["key"])
        if t is None:
            raise RuntimeError(f"ledger holds an unknown task {rec['key']!r}")
        if rec["settings_sha256"] != SETTINGS_SHA256:
            raise RuntimeError(f"ledger entry {rec['key']} was generated under different settings (model, parameters or prompt): refusing to mix; move the old ledger aside deliberately")
        if rec["prompt_sha256"] != t["prompt_sha256"] or rec["context_chunk_ids"] != t["context_chunk_ids"]:
            raise RuntimeError(f"ledger entry {rec['key']} does not match the frozen context/prompt")
        if rec["key"] in done:
            raise RuntimeError(f"ledger holds {rec['key']} twice")
        done[rec["key"]] = rec
    return done


def run_generation(tasks, transport, expected_model, answers_path, attempts_path, window, sleep, progress=print):
    by_key = {t["key"]: t for t in tasks}
    done = load_ledger(answers_path, by_key)
    todo = [t for t in tasks if t["key"] not in done]
    progress(f"resume: {len(done)} of {len(tasks)} answers already on disk; {len(todo)} to generate")
    failed = {}
    for n, t in enumerate(todo, 1):
        rec = generate_one(t, transport, window, sleep, expected_model, attempts_path)
        if rec["status"] == "ok":
            append_jsonl(answers_path, rec)
            done[t["key"]] = rec
            u = rec.get("usage") or {}
            progress(f"  [{n}/{len(todo)}] {t['key']:<14} {t['label']}: ok  {rec['latency_s']:.2f}s  tokens in/out {u.get('prompt_tokens')}/{u.get('completion_tokens')}  finish={rec['finish_reason']}  attempts={rec['attempts']}")
        else:
            failed[t["key"]] = rec
            progress(f"  [{n}/{len(todo)}] {t['key']:<14} {t['label']}: FAILED after {rec['attempts']} attempts ({rec['last_verdict']}: {rec['last_detail']})")
    return done, failed


BASE_Q, RERANK_Q = load_frozen_retrieval()
TASKS = build_tasks(BASE_Q, RERANK_Q)
TASKS_BY_KEY = {t["key"]: t for t in TASKS}
EVAL_PROMPTS_SHA256 = sha256_json([[t["key"], t["prompt_sha256"]] for t in TASKS])
S6_PROMPT_SPEC.write_text(json.dumps(dict(
    purpose="Everything that defines the 48 evaluation prompts, so they can be rebuilt and audited without running anything",
    system_instruction=SYSTEM_INSTRUCTION, context_header=CONTEXT_HEADER, prompt_template=PROMPT_TEMPLATE, generation_config=GENERATION_CONFIG,
    prompt_and_generation_config_sha256=PROMPT_SHA256, message_layout="messages = [system: SYSTEM_INSTRUCTION, user: CONTEXT_HEADER + passages + question]; format_context as in the notebook",
    baseline_file_sha256=BASELINE_FILE_SHA256_PIN, rerank_file_sha256=RERANK_FILE_SHA256_PIN, corpus_sha256=FROZEN_CORPUS_SHA256, eval_prompts_sha256=EVAL_PROMPTS_SHA256,
    tasks=[dict(key=t["key"], query_id=t["query_id"], pipeline=t["pipeline"], label=t["label"], context_chunk_ids=t["context_chunk_ids"], prompt_sha256=t["prompt_sha256"],
                call_position_in_pair=t["call_position_in_pair"]) for t in TASKS]), indent=1, ensure_ascii=False), encoding="utf-8")
print(f"frozen retrieval files verified against their pinned hashes; {len(TASKS)} tasks ({len(TASKS) // 2} queries x 2 pipelines) built; evaluation prompts sha256 = {EVAL_PROMPTS_SHA256}")
print(f"wrote {S6_PROMPT_SPEC} ({sha256_file(S6_PROMPT_SPEC)[:12]}...)")

frozen retrieval files verified against their pinned hashes; 48 tasks (24 queries x 2 pipelines) built; evaluation prompts sha256 = 5a639b1442ea00229e3b9071b9ebd6f57b7bb1a3f6b20bc1304a8248d4dac430
wrote results/nist_stage6_prompt_spec.json (37d96f4734b7...)


## Stage N6d. Offline self-tests of the generation machinery
These tests use a scripted fake transport and temporary files: **no network, no API key, no evaluation question is sent anywhere.**

In [5]:
import tempfile

_n_before = len(SELF_TESTS)


class FakeClock:
    def __init__(self):
        self.t = 1000.0

    def now(self):
        return self.t

    def sleep(self, s):
        self.t += s


def fake_response(status=200, model=None, text="It is described in the passage [nt-001].", finish="stop", headers=None, elapsed=0.5, error=None, body_override="__unset__"):
    model = GEN_MODEL if model is None else model
    if body_override != "__unset__":
        body = body_override
    elif status == 200:
        body = dict(id="chatcmpl-test", model=model, system_fingerprint="fp_test", choices=[dict(index=0, finish_reason=finish, message=dict(role="assistant", content=text))],
                    usage=dict(prompt_tokens=150, completion_tokens=40, total_tokens=190))
    else:
        body = dict(error=dict(message="synthetic error", type="invalid_request_error"))
    return dict(status=status, headers=headers or {}, body=body, error=error, elapsed_s=elapsed)


class FakeTransport:
    """Plays a script of responses in order and records every request body it receives."""

    def __init__(self, script):
        self.script, self.bodies = list(script), []

    def __call__(self, body):
        self.bodies.append(json.loads(json.dumps(body)))
        item = self.script.pop(0) if self.script else fake_response()
        return item(body) if callable(item) else item


def new_run(script, tasks, tmp, secret_key=None, clock=None):
    clock = clock or FakeClock()
    tr = FakeTransport(script)
    window = TokenWindow(TPM_BUDGET, clock=clock.now, sleep=clock.sleep)
    done, failed = run_generation(tasks, tr, GEN_MODEL, Path(tmp) / "answers.jsonl", Path(tmp) / "attempts.jsonl", window, clock.sleep, progress=lambda *_: None)
    return tr, done, failed, clock


# --- request mapping --------------------------------------------------------------------------------------------------------
rq = provider_request(GEN_MODEL, [dict(role="user", content="x")])
check("request: temperature 0, top_p 1, n 1 and the 300-token cap are sent unchanged (mapped to max_completion_tokens)",
      rq["temperature"] == 0.0 and rq["top_p"] == 1.0 and rq["n"] == 1 and rq["max_completion_tokens"] == 300 and rq["stream"] is False)
check("request: the only provider-specific additions are the seed and reasoning_effort", set(rq) - {"model", "messages", "temperature", "top_p", "max_completion_tokens", "n", "stream"} == {"seed", "reasoning_effort"})
per_query_count = collections.Counter(t["query_id"] for t in TASKS)
check("tasks: 48 tasks, both pipelines exactly once for each of the 24 queries, query ids in the preregistered order",
      len(TASKS) == 48 and set(per_query_count.values()) == {2} and list(dict.fromkeys(t["query_id"] for t in TASKS)) == [q["id"] for q in PREREG_Q]
      and all({t["pipeline"] for t in TASKS if t["query_id"] == q["id"]} == {"baseline", "reranked"} for q in PREREG_Q))
persisted_top3 = {("baseline", b["id"]): b["top3_chunk_ids"] for b in BASE_Q} | {("reranked", r["id"]): r["reranked_top3_chunk_ids"] for r in RERANK_Q}
check("tasks: Answer 1 = baseline Top 3 and Answer 2 = re-ranked Top 3, exactly the persisted lists",
      all(t["context_chunk_ids"] == persisted_top3[(t["pipeline"], t["query_id"])] and t["label"] == ("Answer 1" if t["pipeline"] == "baseline" else "Answer 2") for t in TASKS))
check("tasks: within a pair the two requests differ only in the context passages (same model, parameters, instruction and question)",
      all(provider_request(GEN_MODEL, TASKS[i]["messages"]).keys() == provider_request(GEN_MODEL, TASKS[i + 1]["messages"]).keys()
          and TASKS[i]["messages"][0] == TASKS[i + 1]["messages"][0] and TASKS[i]["messages"][1]["content"].split("\n\nQuestion: ")[1] == TASKS[i + 1]["messages"][1]["content"].split("\n\nQuestion: ")[1]
          for i in range(0, 48, 2)))
check("tasks: the call order inside a pair alternates by query (baseline first for even positions, re-ranked first for odd)",
      [t["pipeline"] for t in TASKS[0::2]][:4] == ["baseline", "reranked", "baseline", "reranked"])

# --- classification ---------------------------------------------------------------------------------------------------------
check("classify: a normal 200 is ok", classify(fake_response(), GEN_MODEL)[0] == "ok")
check("classify: an empty 200 answer is no_content", classify(fake_response(text="  "), GEN_MODEL)[0] == "no_content")
check("classify: a different model in the response is fatal (never accept a silent switch)", classify(fake_response(model="some/other-model"), GEN_MODEL)[0] == "fatal")
check("classify: 429/500/503/network error are retried; 400/401/403/404 abort", [classify(fake_response(status=s), GEN_MODEL)[0] for s in (429, 500, 503)] == ["retry"] * 3
      and classify(dict(status=None, headers={}, body=None, error="URLError", elapsed_s=0.1), GEN_MODEL)[0] == "retry" and [classify(fake_response(status=s), GEN_MODEL)[0] for s in (400, 401, 403, 404)] == ["fatal"] * 4)

# --- retry / resume / guards ------------------------------------------------------------------------------------------------
sub = TASKS[:4]
with tempfile.TemporaryDirectory() as tmp:
    tr, done, failed, clock = new_run([fake_response(status=429, headers={"retry-after": "7"}), fake_response(status=503), fake_response(text="First answer [nt-001]."), fake_response(), fake_response(), fake_response()], sub, tmp)
    check("retry: 429 (honouring retry-after) and 503 are retried until success and the answer is recorded", len(done) == 4 and not failed and done[sub[0]["key"]]["attempts"] == 3
          and done[sub[0]["key"]]["answer"] == "First answer [nt-001]." and done[sub[0]["key"]]["paced_wait_s"] >= 7.0 + 4.0)
    check("retry: every retry resent a byte-identical request body", len({json.dumps(b, sort_keys=True) for b in tr.bodies[:3]}) == 1)
    check("audit trail: all 6 attempts (3 for the first answer) are logged, including the failures",
          [json.loads(l)["verdict"] for l in (Path(tmp) / "attempts.jsonl").read_text().splitlines()][:3] == ["retry", "retry", "ok"] and len((Path(tmp) / "attempts.jsonl").read_text().splitlines()) == 6)
    rec = done[sub[0]["key"]]
    check("record: answer, model, settings hash, context ids, usage, timestamps, latency and attempt count are all stored",
          all(k in rec for k in ("answer", "model_requested", "model_returned", "settings_sha256", "context_chunk_ids", "usage", "started_at_utc", "finished_at_utc", "latency_s", "attempts", "finish_reason", "prompt_sha256", "request_sha256")))

with tempfile.TemporaryDirectory() as tmp:
    try:                                                     # a 401 on the 3rd task aborts the run but keeps the first two answers
        new_run([fake_response(), fake_response(), fake_response(status=401)], sub, tmp)
        raise AssertionError("an authentication error did not abort the run")
    except GenerationAbort:
        pass
    saved = [json.loads(l)["key"] for l in (Path(tmp) / "answers.jsonl").read_text().splitlines()]
    check("abort: an authentication error stops the run and keeps the answers already saved", saved == [sub[0]["key"], sub[1]["key"]])
    tr2, done2, failed2, _ = new_run([], sub, tmp)
    check("resume: a second run generates only the missing answers and never regenerates saved ones", len(tr2.bodies) == 2 and len(done2) == 4 and not failed2
          and [json.loads(l)["key"] for l in (Path(tmp) / "answers.jsonl").read_text().splitlines()] == [t["key"] for t in sub])
    tr3, done3, _, _ = new_run([], sub, tmp)
    check("resume: a complete ledger triggers zero API calls", len(tr3.bodies) == 0 and len(done3) == 4)
    lines = (Path(tmp) / "answers.jsonl").read_text().splitlines()
    rec0 = json.loads(lines[0]); rec0["settings_sha256"] = "0" * 64
    (Path(tmp) / "answers.jsonl").write_text(json.dumps(rec0) + "\n" + "\n".join(lines[1:]) + "\n")
    try:
        new_run([], sub, tmp)
        raise AssertionError("a ledger from other settings was accepted")
    except RuntimeError as e:
        check("guard: a ledger written under different settings is refused rather than mixed", "different settings" in str(e))
    (Path(tmp) / "answers.jsonl").write_text("\n".join(lines) + "\n" + '{"key": "A1|base')              # a torn final line
    tr4, done4, _, _ = new_run([], sub, tmp)
    check("resume: a torn final line (killed mid-write) is dropped and nothing else is lost", len(done4) == 4 and len(tr4.bodies) == 0 and len((Path(tmp) / "answers.jsonl").read_text().splitlines()) == 4)

with tempfile.TemporaryDirectory() as tmp:
    try:
        new_run([fake_response(model="some/other-model")], sub, tmp)
        raise AssertionError("a model mismatch did not abort")
    except GenerationAbort:
        check("guard: a response from a different model aborts the run (no silent model switch)", not (Path(tmp) / "answers.jsonl").exists())
with tempfile.TemporaryDirectory() as tmp:
    tr, done, failed, _ = new_run([fake_response(status=503)] * MAX_ATTEMPTS, sub[:1], tmp)
    check("failure: persistent 503 gives up after MAX_ATTEMPTS identical attempts and is reported as failed, never invented", len(tr.bodies) == MAX_ATTEMPTS and not done and sub[0]["key"] in failed)
with tempfile.TemporaryDirectory() as tmp:
    tr, done, failed, _ = new_run([fake_response(text="", finish="length")] * 2, sub[:1], tmp)
    check("failure: an empty answer is retried once with identical settings and then reported as failed", len(tr.bodies) == NO_CONTENT_MAX_ATTEMPTS and not done)
with tempfile.TemporaryDirectory() as tmp:
    tr, done, failed, _ = new_run([fake_response(text="cut off mid", finish="length")], sub[:1], tmp)
    check("truncation: a 'length' finish is saved as-is and flagged truncated (not retried with other settings)", done[sub[0]["key"]]["truncated"] is True and len(tr.bodies) == 1)

# --- pacing ------------------------------------------------------------------------------------------------------------------
clk = FakeClock()
w = TokenWindow(1000, clock=clk.now, sleep=clk.sleep)
w.record(900)
waited = w.wait(500)
check("pacing: a request that would exceed the per-minute budget waits for the window to clear (fake clock)", 59.0 <= waited <= 61.0)
check("pacing: with the real estimates the 48 requests need roughly 8-14 minutes, well inside the daily limit",
      8 * 60 <= sum(estimate_tokens(t["messages"]) + 300 for t in TASKS) / TPM_BUDGET * 60 <= 14 * 60
      and sum(estimate_tokens(t["messages"]) + 300 for t in TASKS) < 0.35 * GROQ_DOCS_SNAPSHOT["free_plan_limits"][GEN_MODEL]["tpd"])

# --- credentials -------------------------------------------------------------------------------------------------------------
with tempfile.TemporaryDirectory() as tmp:
    envp, fake_env = Path(tmp) / ".env", {}
    envp.write_text("# comment\nGROQ_API_KEY=gsk_FAKE_KEY_FOR_TESTS\nOTHER=1\n")
    try:
        load_env_file(envp, fake_env, is_git_ignored=lambda p: False)
        raise AssertionError("an un-ignored .env was read")
    except RuntimeError:
        check("credentials: a .env that git does not ignore is refused", fake_env == {})
    names = load_env_file(envp, fake_env, is_git_ignored=lambda p: True)
    check("credentials: an ignored .env is loaded, only the NAMES are returned, and existing variables are not overridden",
          names == ["GROQ_API_KEY", "OTHER"] and fake_env["GROQ_API_KEY"] == "gsk_FAKE_KEY_FOR_TESTS" and load_env_file(envp, fake_env, is_git_ignored=lambda p: True) == [])
    SECRET = "gsk_FAKE_KEY_FOR_TESTS"
    leaky = dict(status=None, headers={}, body=None, error=redact(f"URLError: connection failed for Bearer {SECRET}", SECRET), elapsed_s=0.1)
    tr, done, failed, _ = new_run([leaky, fake_response()], sub[:1], tmp)
    blob = "".join(p.read_text() for p in Path(tmp).glob("*.jsonl"))
    check("credentials: an error message containing the key is redacted before it is logged, and no saved file contains the key", SECRET not in blob and "[REDACTED]" in blob)
check("credentials: the git-ignore rule for .env holds in this repository", subprocess.run(["git", "check-ignore", "-q", ".env"], capture_output=True).returncode == 0)
check("credentials: request headers are never stored (only an allow-list of rate-limit headers)", set(safe_headers({"Authorization": "Bearer x", "x-ratelimit-remaining-tokens": "5", "Set-Cookie": "c"})) == {"x-ratelimit-remaining-tokens"})
print(f"{len(SELF_TESTS) - _n_before} offline self-tests of the generation machinery passed; nothing was sent over the network")

30 offline self-tests of the generation machinery passed; nothing was sent over the network


## Stage N6e. Gated run: smoke test, then (only when explicitly enabled) the 48 evaluation answers
* `RAG_STAGE6_MODE=off` (default): nothing is sent anywhere.
* `RAG_STAGE6_MODE=smoke`: lists the account's models and sends **three synthetic prompts about an invented village**, with the same instruction, message layout and parameters. No evaluation question is used. It checks the HTTP status, that the returned model id equals the configured one, that the 300-token cap and temperature 0 are accepted, that no reasoning tokens eat the cap, that the answer cites its passage, that an unanswerable question gets the exact refusal, and whether repeating a prompt gives the same text.
* `RAG_STAGE6_MODE=evaluate`: generates the 48 answers, resumably. It **refuses to start** unless a smoke test passed for exactly these settings.

In [6]:
SMOKE_CHUNKS = {"syn-001": dict(page_start=0, text="The Orchard Committee of the (fictional) village of Varnholt meets on the second Tuesday of every month in the old mill.")}
SMOKE_CASES = [
    dict(name="S1 answerable", question="When does the Orchard Committee of Varnholt meet?"),
    dict(name="S2 unanswerable", question="Who chairs the Orchard Committee of Varnholt?"),
    dict(name="S3 repeat of S1", question="When does the Orchard Committee of Varnholt meet?"),
]


def smoke_case_checks(name, res, first_text):
    b = res.get("body") or {}
    ch = (b.get("choices") or [{}])[0]
    msg = ch.get("message") or {}
    text = (msg.get("content") or "").strip()
    usage = b.get("usage") or {}
    reasoning_tokens = (usage.get("completion_tokens_details") or {}).get("reasoning_tokens", 0) or 0
    checks = {
        "HTTP 200": res.get("status") == 200,
        "returned model id equals the configured model id": b.get("model") == GEN_MODEL,
        "finish_reason is 'stop' (not cut by the 300-token cap)": ch.get("finish_reason") == "stop",
        "non-empty answer": bool(text),
        "no reasoning text and no reasoning tokens": not msg.get("reasoning") and reasoning_tokens == 0,
        "completion tokens <= 300": (usage.get("completion_tokens") or 0) <= GENERATION_CONFIG["max_output_tokens"]}
    if name.startswith("S1"):
        checks["answer states 'second Tuesday' and cites [syn-001]"] = "second tuesday" in text.lower() and "syn-001" in text
    if name.startswith("S2"):
        checks["unanswerable question gets the exact refusal sentence"] = text == REFUSAL
    if name.startswith("S3"):
        checks["repeating the prompt returns the identical text (determinism at temperature 0 with a fixed seed)"] = text == first_text
    return checks, text, usage, reasoning_tokens


def run_smoke_test(transport, lister, pause=lambda s: time.sleep(s)):
    models = lister()
    cases, first_text, last_headers, chars = [], None, {}, 0
    for case in SMOKE_CASES:
        msgs = build_messages(case["question"], ["syn-001"], SMOKE_CHUNKS)
        res = transport(provider_request(GEN_MODEL, msgs))
        checks, text, usage, rt = smoke_case_checks(case["name"], res, first_text)
        first_text = text if first_text is None else first_text
        chars += sum(len(m["content"]) for m in msgs)
        last_headers = safe_headers(res.get("headers")) or last_headers
        cases.append(dict(name=case["name"], question=case["question"], answer=text, http_status=res.get("status"), finish_reason=((res.get("body") or {}).get("choices") or [{}])[0].get("finish_reason"),
                          model_returned=(res.get("body") or {}).get("model"), usage=usage, reasoning_tokens=rt, latency_s=round(res["elapsed_s"], 4), error=(res.get("error") or "")[:200] or None,
                          checks=checks, started_at_utc=utc_now()))
        pause(2.0)
    prompt_toks = sum((c["usage"] or {}).get("prompt_tokens") or 0 for c in cases)
    completion_toks = sum((c["usage"] or {}).get("completion_tokens") or 0 for c in cases)
    ratio = (chars / prompt_toks) if prompt_toks else None
    eval_chars = sum(len(m["content"]) for t in TASKS for m in t["messages"])
    free = GROQ_DOCS_SNAPSHOT["free_plan_limits"][GEN_MODEL]
    est_in = int(eval_chars / ratio) if ratio else int(eval_chars * TOKENS_PER_CHAR)
    est_total_upper = est_in + len(TASKS) * GENERATION_CONFIG["max_output_tokens"]
    return dict(
        stage="N6 smoke test: synthetic prompts only; no evaluation question was used", run_at_utc=utc_now(), provider=GEN_PROVIDER, model_requested=GEN_MODEL,
        model_returned=cases[0]["model_returned"], settings_sha256=SETTINGS_SHA256, request_template=REQUEST_TEMPLATE, models_endpoint_lists_the_model=GEN_MODEL in models,
        n_models_listed=len(models), cases=cases, all_checks_passed=GEN_MODEL in models and all(all(c["checks"].values()) for c in cases),
        totals=dict(requests=len(cases), prompt_tokens=prompt_toks, completion_tokens=completion_toks),
        rate_limit_headers_seen=last_headers, observed_chars_per_token=round(ratio, 2) if ratio else None,
        free_tier_projection_for_the_48_evaluation_answers=dict(
            documented_free_limits=free, estimated_input_tokens=est_in, output_tokens_upper_bound=len(TASKS) * GENERATION_CONFIG["max_output_tokens"], total_upper_bound=est_total_upper,
            share_of_daily_token_limit=round(est_total_upper / free["tpd"], 3), requests_vs_daily_request_limit=f"{len(TASKS)} of {free['rpd']}",
            paced_minutes_at_80pct_of_the_per_minute_limit=round(est_total_upper / TPM_BUDGET, 1)))


def require_passing_smoke():
    if not S6_SMOKE.exists():
        raise GenerationAbort("evaluate mode refused: no smoke test on record. Run with RAG_STAGE6_MODE=smoke first.")
    s = json.load(open(S6_SMOKE, encoding="utf-8"))
    if not (s.get("all_checks_passed") and s.get("settings_sha256") == SETTINGS_SHA256 and s.get("model_returned") == GEN_MODEL):
        raise GenerationAbort("evaluate mode refused: the recorded smoke test did not pass, or it was run with other settings/model. Re-run RAG_STAGE6_MODE=smoke.")
    return s


STAGE6_STATUS = dict(mode=STAGE6_MODE, network_used=False)
if STAGE6_MODE == "off":
    print("RAG_STAGE6_MODE=off: no network access in this run, no credentials were read.")
    print(f"  .env present: {ENV_FILE.exists()};  GROQ_API_KEY already exported in the shell: {bool(os.environ.get('GROQ_API_KEY'))}   (status only; no value is ever shown)")
else:
    loaded = load_env_file()
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is not configured. Put it in the git-ignored .env (see README: 'Groq credentials') and re-run; nothing was sent.")
    print(f"credentials: key configured (variables loaded from .env: {loaded or 'none; taken from the shell environment'}); the value is never printed")
    transport, lister = make_groq_transport(api_key), make_groq_model_lister(api_key)
    STAGE6_STATUS["network_used"] = True
    if STAGE6_MODE == "smoke":
        smoke = run_smoke_test(transport, lister)
        assert api_key not in json.dumps(smoke)
        S6_SMOKE.write_text(json.dumps(smoke, indent=1, ensure_ascii=False), encoding="utf-8")
        print(f"smoke test: all checks passed = {smoke['all_checks_passed']}; model listed = {smoke['models_endpoint_lists_the_model']}; returned model = {smoke['model_returned']}")
        for c in smoke["cases"]:
            print(f"  {c['name']:<18} {'PASS' if all(c['checks'].values()) else 'FAIL'}  {c['latency_s']:.2f}s  tokens {c['usage'].get('prompt_tokens')}/{c['usage'].get('completion_tokens')}  answer: {c['answer'][:110]!r}")
            for name, ok in c["checks"].items():
                if not ok:
                    print(f"      FAILED CHECK: {name}")
        print("projection for the 48 evaluation answers:", json.dumps(smoke["free_tier_projection_for_the_48_evaluation_answers"]))
    else:
        require_passing_smoke()
        window = TokenWindow(TPM_BUDGET)
        done, failed = run_generation(TASKS, transport, GEN_MODEL, S6_LEDGER, S6_ATTEMPTS, window, time.sleep)
        assert api_key not in "".join(p.read_text(encoding="utf-8") for p in (S6_LEDGER, S6_ATTEMPTS) if p.exists())
        STAGE6_STATUS.update(answers_on_disk=len(done), failed_this_run=sorted(failed))
        print(f"evaluation generation finished: {len(done)} of {len(TASKS)} answers on disk; failed this run: {sorted(failed) or 'none'}")

credentials: key configured (variables loaded from .env: ['GROQ_API_KEY']); the value is never printed
resume: 0 of 48 answers already on disk; 48 to generate


  [1/48] A1|baseline    Answer 1: ok  0.29s  tokens in/out 655/28  finish=stop  attempts=1


  [2/48] A1|reranked    Answer 2: ok  0.28s  tokens in/out 592/28  finish=stop  attempts=1


  [3/48] A2|reranked    Answer 2: ok  0.27s  tokens in/out 513/26  finish=stop  attempts=1


  [4/48] A2|baseline    Answer 1: ok  0.31s  tokens in/out 498/29  finish=stop  attempts=1


...

  [44/48] O4|baseline    Answer 1: ok  0.28s  tokens in/out 674/28  finish=stop  attempts=1


  [45/48] O5|baseline    Answer 1: ok  0.54s  tokens in/out 452/33  finish=stop  attempts=1


  [46/48] O5|reranked    Answer 2: ok  0.31s  tokens in/out 579/51  finish=stop  attempts=1


  [47/48] O6|reranked    Answer 2: ok  0.30s  tokens in/out 706/33  finish=stop  attempts=1


  [48/48] O6|baseline    Answer 1: ok  0.28s  tokens in/out 604/33  finish=stop  attempts=1
evaluation generation finished: 48 of 48 answers on disk; failed this run: none


## Stage N6f. Assembly: the 48 answers, the per-query comparison and the LLM latency summary
Everything here is read from the ledger on disk (nothing is regenerated). Answer quality is **kept separate from retrieval success**: the table puts the frozen retrieval outcomes next to the answers, plus a few *objective* answer checks (does it cite only the passages it was given, is it the refusal sentence, was it cut by the token cap). Whether an answer is *correct* is a human judgement made later on a review sheet; no LLM judge is used as ground truth.

In [7]:
import csv
import math


def cited_ids(text):
    return list(dict.fromkeys(re.findall(r"nt-\d{3}", text or "")))


def summarize_stats(values_ms):
    """n, mean, median, sample standard deviation, nearest-rank 95th percentile, min, max (all in milliseconds)."""
    if not values_ms:
        return dict(n=0)
    xs = sorted(values_ms)
    p95 = xs[math.ceil(0.95 * len(xs)) - 1]
    return dict(n=len(xs), mean_ms=round(statistics.fmean(xs), 2), median_ms=round(statistics.median(xs), 2), stdev_ms=round(statistics.stdev(xs), 2) if len(xs) > 1 else 0.0,
                p95_ms=round(p95, 2), min_ms=round(xs[0], 2), max_ms=round(xs[-1], 2))


def summarize_ms(values_s):
    return summarize_stats([v * 1000.0 for v in values_s])


def assemble_stage6():
    if not S6_LEDGER.exists():
        print("Stage 6: no evaluation answers exist yet (the generation has not been run); nothing to assemble.")
        return None
    done = load_ledger(S6_LEDGER, TASKS_BY_KEY)
    attempts = collections.defaultdict(list)
    if S6_ATTEMPTS.exists():
        for ln in S6_ATTEMPTS.read_text(encoding="utf-8").splitlines():
            if ln.strip():
                a = json.loads(ln)
                attempts[a["key"]].append(a)
    order = {"baseline": 0, "reranked": 1}
    qpos = {q["id"]: i for i, q in enumerate(PREREG_Q)}
    recs = []
    for t in sorted(TASKS, key=lambda t: (qpos[t["query_id"]], order[t["pipeline"]])):
        r = done.get(t["key"])
        if r is None:
            last = attempts[t["key"]][-1] if attempts[t["key"]] else None
            r = dict(status="missing", key=t["key"], query_id=t["query_id"], pipeline=t["pipeline"], label=t["label"], context_chunk_ids=t["context_chunk_ids"],
                     attempts=len(attempts[t["key"]]), last_verdict=last["verdict"] if last else None, last_detail=last["detail"] if last else "not attempted")
        recs.append(r)
    ok = [r for r in recs if r["status"] == "ok"]
    S6_ANSWERS.write_text(json.dumps(dict(
        stage="N6 answer generation: Answer 1 = baseline Top 3 (Bi-Encoder), Answer 2 = re-ranked Top 3 (Cross-Encoder); same model, prompt layout and parameters",
        provider=GEN_PROVIDER, model=GEN_MODEL, request_template=REQUEST_TEMPLATE, settings_sha256=SETTINGS_SHA256, prompt_and_generation_config_sha256=PROMPT_SHA256,
        eval_prompts_sha256=EVAL_PROMPTS_SHA256, baseline_file_sha256=BASELINE_FILE_SHA256_PIN, rerank_file_sha256=RERANK_FILE_SHA256_PIN, corpus_sha256=FROZEN_CORPUS_SHA256,
        counts=dict(expected=len(TASKS), ok=len(ok), missing_or_failed=len(TASKS) - len(ok), truncated=sum(r.get("truncated", False) for r in ok),
                    total_attempts=sum(len(v) for v in attempts.values()), failed_attempts=sum(a["verdict"] != "ok" for v in attempts.values() for a in v)),
        records=recs), indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"assembled {len(ok)} of {len(TASKS)} answers into {S6_ANSWERS.name}; missing/failed: {[r['key'] for r in recs if r['status'] != 'ok'] or 'none'}")

    qindex = {q["id"]: i for i, q in enumerate(PREREG_Q)}
    rows = []
    for q in PREREG_Q:
        i = qindex[q["id"]]
        b, rr = BASE_Q[i], RERANK_Q[i]
        a1 = next(r for r in recs if r["query_id"] == q["id"] and r["pipeline"] == "baseline")
        a2 = next(r for r in recs if r["query_id"] == q["id"] and r["pipeline"] == "reranked")
        row = dict(query_id=q["id"], category=q["category"], question=q["question"], gold_chunk_ids=" ".join(q["gold_chunk_ids"]),
                   baseline_top3=" ".join(b["top3_chunk_ids"]), reranked_top3=" ".join(rr["reranked_top3_chunk_ids"]),
                   best_gold_rank_baseline=b["lenient"]["best_gold_rank"], best_gold_rank_reranked=rr["lenient"]["best_gold_rank"],
                   lenient_baseline=b["lenient"]["outcome"], lenient_reranked=rr["lenient"]["outcome"], lenient_movement=rr["lenient_movement"],
                   strict_baseline=b["strict"]["outcome"], strict_reranked=rr["strict"]["outcome"], strict_baseline_top3_evidence=b["strict"]["top3_evidence"],
                   strict_reranked_top3_evidence=rr["strict"]["top3_evidence"], strict_movement=rr["strict_movement"])
        for tag, a in (("a1", a1), ("a2", a2)):
            txt = a.get("answer")
            cites = cited_ids(txt)
            row |= {f"{tag}_status": a["status"], f"{tag}_answer": txt or "", f"{tag}_cited_ids": " ".join(cites),
                    f"{tag}_cites_only_given_passages": (set(cites) <= set(a["context_chunk_ids"])) if txt else "",
                    f"{tag}_is_refusal": (txt.strip() == REFUSAL) if txt else "", f"{tag}_truncated": a.get("truncated", "") if txt else "",
                    f"{tag}_finish_reason": a.get("finish_reason", ""), f"{tag}_latency_s": a.get("latency_s", ""),
                    f"{tag}_completion_tokens": (a.get("usage") or {}).get("completion_tokens", "")}
        rows.append(row)
    with open(S6_COMPARISON, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print(f"wrote {S6_COMPARISON.name} ({len(rows)} rows x {len(rows[0])} columns)\n")

    print("=" * 110)
    for row in rows:
        print(f"{row['query_id']} [{row['category']}] {row['question']}")
        print(f"   gold {row['gold_chunk_ids']} | gold rank {row['best_gold_rank_baseline']} -> {row['best_gold_rank_reranked']} | lenient {row['lenient_baseline']} -> {row['lenient_reranked']} | strict {row['strict_baseline']} -> {row['strict_reranked']}")
        for tag, name, ctx in (("a1", "Answer 1 (baseline)", "baseline_top3"), ("a2", "Answer 2 (re-ranked)", "reranked_top3")):
            flags = [f for f, on in (("REFUSAL", row[f"{tag}_is_refusal"] is True), ("TRUNCATED", row[f"{tag}_truncated"] is True), ("CITES-OUTSIDE-CONTEXT", row[f"{tag}_cites_only_given_passages"] is False)) if on]
            print(f"   {name}  context [{row[ctx]}]  {' '.join(flags)}\n      {row[f'{tag}_answer'] or '(' + row[f'{tag}_status'] + ')'}")
    print("\nLLM generation latency (wall clock of the successful HTTP call, includes network; excludes pacing sleeps and retries)")
    lat = {}
    for pipe in ("baseline", "reranked"):
        lat[pipe] = summarize_ms([r["latency_s"] for r in ok if r["pipeline"] == pipe])
        print(f"   {pipe:<9} {lat[pipe]}")
    return dict(records=recs, rows=rows, latency=lat)


STAGE6_RESULT = assemble_stage6()

assembled 48 of 48 answers into nist_stage6_answers.json; missing/failed: none
wrote nist_stage6_comparison.csv (24 rows x 34 columns)

A1 [keyword trap] Which risk assessment orientation begins with the consequences we care about and our most critical assets, rather than with attackers or weaknesses?
   gold nt-063 | gold rank 3 -> 2 | lenient success -> success | strict success -> success
   Answer 1 (baseline)  context [nt-064 nt-020 nt-063]  
      The asset/impact-oriented approach begins with the identification of impacts or consequences of concern and critical assets. [nt-063]
   Answer 2 (re-ranked)  context [nt-064 nt-063 nt-238]  
      The asset/impact-oriented approach begins with the identification of impacts or consequences of concern and critical assets. [nt-063]
A2 [keyword trap] Which term describes different experts arriving at the same results from the same data?
   gold nt-235 | gold rank 1 -> 1 | lenient success -> success | strict success -> success
   Answer 1 (b

## Stage N6g. Offline self-tests of the smoke test, the evaluate-mode gate and the assembly
Fake transport and temporary files again: **no network, no credentials, the real result files are not touched.**

In [8]:
import contextlib

_n_before = len(SELF_TESTS)
import io

GOOD_S1 = "The committee meets on the second Tuesday of every month [syn-001]."


def smoke_script(s1=GOOD_S1, s2=REFUSAL, s3=None, model=None, finish="stop"):
    return [fake_response(text=s1, model=model, finish=finish), fake_response(text=s2, model=model, finish=finish), fake_response(text=s1 if s3 is None else s3, model=model, finish=finish)]


def run_fake_smoke(script, listed=None):
    return run_smoke_test(FakeTransport(script), lambda: [GEN_MODEL, "openai/gpt-oss-120b"] if listed is None else listed, pause=lambda s: None)


good = run_fake_smoke(smoke_script())
check("smoke test: a healthy model passes every check and reports the free-tier projection for the 48 answers",
      good["all_checks_passed"] and good["models_endpoint_lists_the_model"] and good["model_returned"] == GEN_MODEL
      and good["free_tier_projection_for_the_48_evaluation_answers"]["share_of_daily_token_limit"] < 0.35 and good["totals"]["requests"] == 3)
check("smoke test: it uses only synthetic prompts (no evaluation question text appears in them)",
      all(not any(q["question"] in c["question"] for q in PREREG_Q) for c in good["cases"]) and all(SMOKE_CHUNKS["syn-001"]["text"] not in CHUNK_BY_ID[c]["text"] for c in list(CHUNK_BY_ID)[:50]))
check("smoke test: a wrong refusal, a truncated answer, a different model, a missing model listing or leaked reasoning each FAIL the test",
      not run_fake_smoke(smoke_script(s2="Maybe the mayor."))["all_checks_passed"]
      and not run_fake_smoke(smoke_script(finish="length"))["all_checks_passed"]
      and not run_fake_smoke(smoke_script(model="some/other-model"))["all_checks_passed"]
      and not run_fake_smoke(smoke_script(), listed=["openai/gpt-oss-120b"])["all_checks_passed"]
      and not run_fake_smoke(smoke_script(s3="A different phrasing [syn-001] second Tuesday."))["all_checks_passed"]
      and not run_fake_smoke([fake_response(text=GOOD_S1, body_override=dict(id="x", model=GEN_MODEL, choices=[dict(finish_reason="stop", message=dict(content=GOOD_S1, reasoning="thinking..."))],
                                                                              usage=dict(prompt_tokens=300, completion_tokens=40)))] * 3)["all_checks_passed"])

_saved_paths = {n: globals()[n] for n in ("S6_SMOKE", "S6_LEDGER", "S6_ATTEMPTS", "S6_ANSWERS", "S6_COMPARISON")}
try:
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        S6_SMOKE = tmp / "smoke.json"
        try:
            require_passing_smoke()
            raise AssertionError("evaluate mode ran without a smoke test")
        except GenerationAbort:
            pass
        S6_SMOKE.write_text(json.dumps(dict(good, all_checks_passed=False)))
        gate_failing = False
        try:
            require_passing_smoke()
        except GenerationAbort:
            gate_failing = True
        S6_SMOKE.write_text(json.dumps(dict(good, settings_sha256="0" * 64)))
        gate_other_settings = False
        try:
            require_passing_smoke()
        except GenerationAbort:
            gate_other_settings = True
        S6_SMOKE.write_text(json.dumps(good))
        check("gate: evaluate mode refuses without a smoke test, with a failed one, or with one recorded under other settings; it accepts a passing one for these settings",
              gate_failing and gate_other_settings and require_passing_smoke()["all_checks_passed"])

        S6_LEDGER, S6_ATTEMPTS, S6_ANSWERS, S6_COMPARISON = tmp / "a.jsonl", tmp / "att.jsonl", tmp / "ans.json", tmp / "cmp.csv"
        with contextlib.redirect_stdout(io.StringIO()) as buf:
            before = assemble_stage6()
        check("assembly: with no ledger it says so and writes nothing", before is None and not S6_ANSWERS.exists() and "no evaluation answers exist yet" in buf.getvalue())
        window = TokenWindow(TPM_BUDGET, clock=(c := FakeClock()).now, sleep=c.sleep)
        done, failed = run_generation(TASKS[:40], FakeTransport([]), GEN_MODEL, S6_LEDGER, S6_ATTEMPTS, window, c.sleep, progress=lambda *_: None)
        with contextlib.redirect_stdout(io.StringIO()):
            partial = assemble_stage6()
        saved = json.load(open(S6_ANSWERS))
        check("assembly: a partial ledger (40 of 48) is reported honestly, the missing answers are listed as missing and never filled in",
              saved["counts"]["ok"] == 40 and saved["counts"]["missing_or_failed"] == 8 and sum(r["status"] == "missing" for r in saved["records"]) == 8 and len(saved["records"]) == 48)
        done, failed = run_generation(TASKS, FakeTransport([]), GEN_MODEL, S6_LEDGER, S6_ATTEMPTS, window, c.sleep, progress=lambda *_: None)
        with contextlib.redirect_stdout(io.StringIO()):
            full = assemble_stage6()
        saved = json.load(open(S6_ANSWERS))
        rows = list(csv.DictReader(open(S6_COMPARISON, encoding="utf-8")))
        check("assembly: the full ledger gives 48 records in query order (Answer 1 then Answer 2) and a 24-row comparison with retrieval outcomes beside the answers",
              saved["counts"]["ok"] == 48 and [r["label"] for r in saved["records"][:4]] == ["Answer 1", "Answer 2", "Answer 1", "Answer 2"] and len(rows) == 24
              and rows[0]["query_id"] == PREREG_Q[0]["id"] and rows[0]["lenient_baseline"] == BASE_Q[0]["lenient"]["outcome"] and "a2_answer" in rows[0])
        check("assembly: objective answer checks flag a citation outside the supplied passages (the fake answers cite nt-001 regardless of context)",
              any(r["a1_cites_only_given_passages"] == "False" for r in rows) and all(r["a1_is_refusal"] == "False" for r in rows))
        check("assembly: latency is summarised per pipeline with n, mean, median, standard deviation, 95th percentile, min and max",
              set(full["latency"]["baseline"]) == {"n", "mean_ms", "median_ms", "stdev_ms", "p95_ms", "min_ms", "max_ms"} and full["latency"]["baseline"]["n"] == 24)
finally:
    globals().update(_saved_paths)
check("tests used temporary files only: the real Stage 6 result paths are untouched", all(globals()[n] == v for n, v in _saved_paths.items()))
print(f"{len(SELF_TESTS) - _n_before} offline self-tests of the smoke test, gating and assembly passed; nothing was sent over the network")

10 offline self-tests of the smoke test, gating and assembly passed; nothing was sent over the network


## Stage N6h. Human-review sheet: Answer 1 beside Answer 2, with the evidence each one was given
Written from the ledger and the frozen retrieval files, for **you** to grade. For every query it shows the preregistered evidence, which chunks each pipeline put in front of the LLM (with their baseline and re-ranked ranks),
the full text of those chunks, both answers with their citations, and a few **objective format checks**. **Those checks are not evidence that an answer is factually correct**: an answer can cite the right passage and still be wrong, or be right without a citation.
The grading columns are left blank on purpose.
If `results/nist_stage6_grading_guide.json` exists, a **draft expected-answer key** is shown for every query (written from the frozen passages, the preregistered evidence and overlay v1.1, not from any answer) and the structured grading fields are appended to the CSV. Existing lines and columns are never altered.

In [9]:
S6_REVIEW_MD, S6_REVIEW_CSV = OUT_DIR / "nist_stage6_human_review.md", OUT_DIR / "nist_stage6_human_review.csv"
S6_GUIDE = OUT_DIR / "nist_stage6_grading_guide.json"


def sentence_count(text):
    """Approximate: sentence-final punctuation followed by a capital letter, bracket or quote. Only used to flag the 'at most three sentences' instruction."""
    return len([s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z\[(\"“])", (text or "").strip()) if s.strip()])


def objective_checks(rec, gold_ids):
    txt = rec.get("answer") or ""
    cites = cited_ids(txt)
    ok = rec["status"] == "ok"
    return dict(generated=ok, finish_reason_stop=ok and rec.get("finish_reason") == "stop", cites_a_passage=bool(cites),
                cites_only_supplied_passages=ok and bool(cites) and set(cites) <= set(rec["context_chunk_ids"]), cites_a_preregistered_gold_chunk=bool(set(cites) & set(gold_ids)),
                is_the_refusal_sentence=ok and txt.strip() == REFUSAL, approx_sentences=sentence_count(txt) if ok else None, within_three_sentence_limit=ok and sentence_count(txt) <= 3)


def build_review_sheet(recs, md_path, csv_path, guide_path=None):
    by_key = {r["key"]: r for r in recs}
    guide = json.load(open(guide_path, encoding="utf-8")) if guide_path and Path(guide_path).exists() else None
    gq = {g["id"]: g for g in guide["queries"]} if guide else {}
    prereg = {q["id"]: q for q in json.load(open(OUT_DIR / "nist_preregistration.json", encoding="utf-8"))["queries"]}
    md = ["# Human-review sheet: Answer 1 (baseline Top 3) vs Answer 2 (re-ranked Top 3)", "",
          f"Model `{GEN_MODEL}` via {GEN_PROVIDER}; identical prompt and parameters for both answers (settings sha256 `{SETTINGS_SHA256[:16]}...`). Retrieval files pinned: baseline `{BASELINE_FILE_SHA256_PIN[:12]}...`, re-ranked `{RERANK_FILE_SHA256_PIN[:12]}...`.", "",
          "**How to read it.** *Frozen retrieval outcome* is the preregistered Stage 4/5 result and is not changed by anything here. *Objective checks* only test format and consistency (does the answer cite a passage it was given, is it the refusal sentence, was it cut off, does it respect the three-sentence limit). "
          "**They are not proof that an answer is factually correct.** Judge correctness yourself against the passages shown; the grading fields are blank.", ""]
    if guide:
        d = guide["definitions"]
        md += ["## Grading legend (applies to every query)", "", f"*Key status: {guide['status']}.* Guide sha256 `{sha256_file(guide_path)[:16]}...`. Authored by {guide['authoring']['author']}.", "",
               "| CSV field (Answer 1 / Answer 2) | allowed values | judged against |", "|---|---|---|",
               f"| `a1_correct` / `a2_correct` | {' / '.join(d['correctness']['values'])} | {d['correctness']['reference']} |",
               f"| `a1_supported_by_passages` / `a2_supported_by_passages` | {' / '.join(d['evidence_support']['values'])} | {d['evidence_support']['reference']} |",
               f"| `a1_complete` / `a2_complete` | {' / '.join(d['completeness']['values'])} | {d['completeness']['reference']} |",
               f"| `a1_unsupported_claims` / `a2_unsupported_claims` | {' / '.join(d['unsupported_claims']['values'])} (+ details in `*_unsupported_claims_notes`) | {d['unsupported_claims']['rule']} |",
               f"| `a1_overall_judgment` / `a2_overall_judgment` (+ `*_justification`) | {' / '.join(d['overall_judgment']['values'])} | {d['overall_judgment']['suggestion']} |",
               f"| `better` | {' / '.join(d['paired_judgment']['values'])} | {d['paired_judgment']['rule']} |", "", "**Conventions:**"] + [f"* {c}" for c in guide["conventions"]] + ["", "---", ""]
    rows = []
    for i, q in enumerate(PREREG_Q):
        b, r = BASE_Q[i], RERANK_Q[i]
        assert b["id"] == r["id"] == q["id"]
        a1, a2 = by_key[f"{q['id']}|baseline"], by_key[f"{q['id']}|reranked"]
        pq = prereg[q["id"]]
        evid = [" + ".join(g) if isinstance(g, list) else str(g) for g in (pq["evidence"] if isinstance(pq["evidence"], list) else [pq["evidence"]])]      # each group = the phrases of one gold chunk / joint set
        brow, rrow = {x["chunk_id"]: x for x in b["top10"]}, {x["chunk_id"]: x for x in r["reranked_top10"]}
        ctx = list(dict.fromkeys(a1["context_chunk_ids"] + a2["context_chunk_ids"]))
        c1, c2 = objective_checks(a1, q["gold_chunk_ids"]), objective_checks(a2, q["gold_chunk_ids"])
        same = (a1.get("answer") or "").strip() == (a2.get("answer") or "").strip() and a1["status"] == a2["status"] == "ok"
        md += [f"## {q['id']}  ({q['category']})", "", f"**Question:** {q['question']}", "",
               f"**Preregistered gold chunk(s):** {', '.join(q['gold_chunk_ids'])}", "", "**Preregistered evidence phrase(s):**"] + [f"> {e}" for e in evid] + [
               "", f"**Frozen retrieval outcome:** lenient {b['lenient']['outcome']} -> {r['lenient']['outcome']} (best gold rank {b['lenient']['best_gold_rank']} -> {r['lenient']['best_gold_rank']}; {r['lenient_movement']}); "
               f"strict {b['strict']['outcome']} -> {r['strict']['outcome']} (Top-3 evidence {b['strict']['top3_evidence']} -> {r['strict']['top3_evidence']}; {r['strict_movement']})", "",
               "| chunk | Answer 1 context? (baseline rank) | Answer 2 context? (re-ranked rank, was baseline rank) | preregistered gold | strict role |", "|---|---|---|---|---|"]
        for cid in ctx:
            in1, in2 = cid in a1["context_chunk_ids"], cid in a2["context_chunk_ids"]
            md.append(f"| {cid} | {'yes (rank ' + str(brow[cid]['rank']) + ')' if in1 else 'no (was rank ' + str(brow[cid]['rank']) + ')'} | "
                      f"{'yes (rank ' + str(rrow[cid]['rank']) + ', was ' + str(rrow[cid]['original_rank']) + ')' if in2 else 'no (re-ranked rank ' + str(rrow[cid]['rank']) + ')'} | "
                      f"{'GOLD' if cid in q['gold_chunk_ids'] else ''} | {brow[cid]['strict_role']} |")
        md += ["", "**Passages shown to the LLM (union of the two contexts):**", ""]
        for cid in ctx:
            md += [f"> **{cid}** (PDF page {CHUNK_BY_ID[cid]['page_start']})"] + [f"> {ln}" for ln in CHUNK_BY_ID[cid]["text"].splitlines()] + [">"]
        if guide:
            g = gq[q["id"]]
            assert g["question"] == q["question"] and g["preregistered_gold_chunk_ids"] == q["gold_chunk_ids"], f"{q['id']}: the guide does not match the frozen question / gold ids"
            md += ["", "**Grading guide (draft key; derived from the frozen source, not from any answer)**", "", f"* **Expected answer:** {g['expected_answer']}", f"* **Expected polarity:** {g['expected_polarity']}  |  type: {g['answer_type']}",
                   "* **Required points:** " + " ".join(f"({i}) {p}" for i, p in enumerate(g["required_points"], 1)),
                   f"* **Must it mention an exception / qualification / negation?** {'yes: ' + g['qualification_kind'] + ': ' + g['qualification'] + ' (affects ' + g['qualification_affects'] + ')' if g['must_mention_qualification'] else 'no'}",
                   f"* **Acceptable wording:** {g['acceptable_wording']}", "* **Would count as an unsupported claim / error:** " + " ".join(f"({i}) {u}" for i, u in enumerate(g["unsupported_claim_examples"], 1)),
                   "* **Supporting passages:** " + "; ".join(f"`{p['chunk']}`: \"{p['quote']}\"" for p in g["supporting_passages"])]
            if g["grader_notes"]:
                md.append(f"* **Grader notes:** {g['grader_notes']}")
        for tag, name, rec, ck, ctxkey in (("a1", "Answer 1 (baseline)", a1, c1, "baseline"), ("a2", "Answer 2 (re-ranked)", a2, c2, "reranked")):
            md += ["", f"**{name}**  | context order: {', '.join(rec['context_chunk_ids'])}", "", f"> {rec.get('answer') or '(not generated: ' + str(rec.get('last_detail')) + ')'}", "",
                   f"*Objective checks (format only):* cites a passage: {ck['cites_a_passage']}; only supplied passages: {ck['cites_only_supplied_passages']}; cites a gold chunk: {ck['cites_a_preregistered_gold_chunk']}; "
                   f"refusal sentence: {ck['is_the_refusal_sentence']}; finish_reason stop: {ck['finish_reason_stop']}; approx. sentences: {ck['approx_sentences']} (limit 3: {ck['within_three_sentence_limit']}); "
                   f"latency {rec.get('latency_s')} s; tokens in/out {(rec.get('usage') or {}).get('prompt_tokens')}/{(rec.get('usage') or {}).get('completion_tokens')}"]
        md += ["", f"**Answers textually identical:** {'yes' if same else 'no'}", "",
               "**Your grading** (blank):  Answer 1 correct? ____  supported by its passages? ____  complete? ____  unsupported claims: ____   |   Answer 2 correct? ____  supported? ____  complete? ____  unsupported claims: ____   |   better: A1 / A2 / tie  ____", ""] + (["*Enter the structured grades in the CSV row for this query (allowed values: see the legend at the top).*", ""] if guide else []) + ["---", ""]
        row = dict(query_id=q["id"], category=q["category"], question=q["question"], gold_chunk_ids=" ".join(q["gold_chunk_ids"]), preregistered_evidence=" || ".join(evid),
                   baseline_context=" ".join(a1["context_chunk_ids"]), reranked_context=" ".join(a2["context_chunk_ids"]),
                   baseline_best_gold_rank=b["lenient"]["best_gold_rank"], reranked_best_gold_rank=r["lenient"]["best_gold_rank"], lenient_baseline=b["lenient"]["outcome"], lenient_reranked=r["lenient"]["outcome"],
                   strict_baseline=b["strict"]["outcome"], strict_reranked=r["strict"]["outcome"], answers_identical=same)
        for tag, rec, ck in (("a1", a1, c1), ("a2", a2, c2)):
            row |= {f"{tag}_answer": rec.get("answer") or "", f"{tag}_cited_ids": " ".join(cited_ids(rec.get("answer"))), **{f"{tag}_chk_{k}": v for k, v in ck.items()}}
        row |= {f: "" for f in ("a1_correct", "a1_supported_by_passages", "a1_complete", "a1_unsupported_claims", "a2_correct", "a2_supported_by_passages", "a2_complete", "a2_unsupported_claims", "better", "reviewer_notes")}
        if guide:
            g = gq[q["id"]]
            row |= {f: "" for f in ("a1_unsupported_claims_notes", "a1_overall_judgment", "a1_justification", "a2_unsupported_claims_notes", "a2_overall_judgment", "a2_justification")}
            row |= dict(guide_expected_answer=g["expected_answer"], guide_expected_polarity=g["expected_polarity"], guide_required_points=" || ".join(g["required_points"]),
                        guide_must_mention_qualification="yes" if g["must_mention_qualification"] else "no", guide_qualification_kind=g["qualification_kind"], guide_qualification=g["qualification"],
                        guide_qualification_affects=g["qualification_affects"], guide_acceptable_wording=g["acceptable_wording"], guide_unsupported_claim_examples=" || ".join(g["unsupported_claim_examples"]),
                        guide_supporting_passages=" || ".join(f"{p['chunk']}: {p['quote']}" for p in g["supporting_passages"]), guide_grader_notes=g["grader_notes"])
        rows.append(row)
    Path(md_path).write_text("\n".join(md) + "\n", encoding="utf-8")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    return rows


if STAGE6_RESULT and all(r["status"] == "ok" for r in STAGE6_RESULT["records"]):
    REVIEW_ROWS = build_review_sheet(STAGE6_RESULT["records"], S6_REVIEW_MD, S6_REVIEW_CSV, S6_GUIDE)
    print(f"wrote {S6_REVIEW_MD.name} ({S6_REVIEW_MD.stat().st_size // 1024} KB) and {S6_REVIEW_CSV.name} ({len(REVIEW_ROWS)} rows, grading columns blank)")
else:
    REVIEW_ROWS = None
    print("review sheet not written: it needs all 48 answers (none yet, or some are missing/failed).")

wrote nist_stage6_human_review.md (163 KB) and nist_stage6_human_review.csv (24 rows, grading columns blank)


## Stage N7. Latency of the retrieval and re-ranking steps
**Protocol (fixed before measuring).** `time.perf_counter`, wall clock, CPU only, **single-threaded** (`OMP_NUM_THREADS=1`, forced by the macOS OpenMP crash noted in the README), one query at a time, models and index already loaded.
* Components: (1) query embedding, (2) FAISS exact Top-10 search, (3) Cross-Encoder scoring of the ten (query, chunk) pairs, (4) sorting the ten logits. **Pipeline 1** = (1)+(2). **Pipeline 2** = (1)+(2)+(3)+(4); its extra cost over Pipeline 1 is the re-ranking overhead.
* 3 discarded warm-up passes, then 20 timed passes over all 24 questions, giving n = 480 samples per component. Reported: mean, median, sample standard deviation, 95th percentile (nearest rank), min and max.
* One-time costs (loading each model, reading the index) are timed separately from local files and are **not** part of the steady-state numbers. Model download time is never included.
* Every timed pass is also checked against the persisted Stage 4/5 results (same Top 10, same re-ranked order); a mismatch would invalidate the run.
* Supplementary: scoring **all 305 chunks** with the Cross-Encoder for each question (3 passes) to measure what skipping the Top-10 pre-filter would cost on this tiny corpus; larger sizes are linear **projections**, labelled as such, not measurements.
* LLM generation latency is a different kind of measurement (remote service, network included) and is summarised separately above from the 48 recorded calls; the two are never added together as if they shared conditions.
* `RAG_LATENCY_MODE=measure` runs the measurement and writes `results/nist_latency_retrieval.json` (it never touches the frozen retrieval files); the default `report` mode only displays a saved measurement.

In [10]:
import platform

LATENCY_MODE = os.environ.get("RAG_LATENCY_MODE", "report")
assert LATENCY_MODE in ("report", "measure"), "RAG_LATENCY_MODE must be report or measure"
LATENCY_PATH = OUT_DIR / "nist_latency_retrieval.json"
WARMUP_PASSES, TIMED_PASSES, FULL_COLLECTION_PASSES = 3, 20, 3


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=10).stdout.strip() or None
    except Exception:
        return None


def machine_info():
    mem = _sh(["sysctl", "-n", "hw.memsize"])
    return dict(cpu=_sh(["sysctl", "-n", "machdep.cpu.brand_string"]), logical_cpus=os.cpu_count(), memory_gb=round(int(mem) / 2**30, 1) if mem else None,
                os=platform.platform(), python=platform.python_version(), omp_num_threads=os.environ.get("OMP_NUM_THREADS"), torch_threads=torch.get_num_threads(),
                faiss_threads=faiss.omp_get_max_threads(), device="cpu", load_average_1_5_15=[round(x, 2) for x in os.getloadavg()],
                caveat="a general-purpose laptop/desktop with other processes running; thermal state and background load were not controlled")


def timed_pipeline(question):
    t0 = time.perf_counter()
    qv = model.encode([question], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
    t1 = time.perf_counter()
    _, ids = index.search(qv, K_INIT)
    t2 = time.perf_counter()
    cand = [CHUNKS[int(i)]["chunk_id"] for i in ids[0]]
    pairs = [(question, CHUNK_BY_ID[c]["text"]) for c in cand]
    t3 = time.perf_counter()
    logits = [float(x) for x in ce.predict(pairs, batch_size=K_INIT, show_progress_bar=False, activation_fn=torch.nn.Identity())]
    t4 = time.perf_counter()
    reranked = [cand[i] for i in sorted(range(K_INIT), key=lambda i: (-logits[i], i))]
    t5 = time.perf_counter()
    ms = lambda a, b: (b - a) * 1000.0
    return dict(embed=ms(t0, t1), search=ms(t1, t2), ce_score=ms(t3, t4), sort=ms(t4, t5), pipeline1=ms(t0, t2), pipeline2=ms(t0, t5), overhead=ms(t2, t5)), cand, reranked


def measure_latency():
    base_ids = {b["id"]: [r["chunk_id"] for r in b["top10"]] for b in BASE_Q}
    rer_ids = {r["id"]: [x["chunk_id"] for x in r["reranked_top10"]] for r in RERANK_Q}
    info_before = machine_info()
    # one-time initialisation (local files only; separate from steady state)
    init = {}
    t = time.perf_counter(); _m = SentenceTransformer(EMBED_MODEL, device="cpu", local_files_only=True); init["load_bi_encoder_s"] = round(time.perf_counter() - t, 3); del _m
    t = time.perf_counter(); _c = CrossEncoder(CE_NAME, device="cpu", local_files_only=True); init["load_cross_encoder_s"] = round(time.perf_counter() - t, 3); del _c
    t = time.perf_counter(); _i = faiss.read_index(str(INDEX_PATH)); init["read_faiss_index_s"] = round(time.perf_counter() - t, 3); del _i
    t = time.perf_counter(); _e = model.encode(texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False); init["embed_305_chunks_s"] = round(time.perf_counter() - t, 3); del _e

    qs = [(q["id"], q["question"]) for q in PREREG_Q]
    first_call = timed_pipeline(qs[0][1])[0]                       # first call of this block (models are already loaded, so not a true cold start)
    for _ in range(WARMUP_PASSES):
        for _, qq in qs:
            timed_pipeline(qq)
    samples, per_q, mismatches = collections.defaultdict(list), collections.defaultdict(lambda: collections.defaultdict(list)), 0
    pass_means = collections.defaultdict(list)
    for p in range(TIMED_PASSES):
        this_pass = collections.defaultdict(list)
        for qid, qq in qs:
            comp, cand, reranked = timed_pipeline(qq)
            mismatches += int(cand != base_ids[qid]) + int(reranked != rer_ids[qid])
            for k, v in comp.items():
                samples[k].append(v); per_q[qid][k].append(v); this_pass[k].append(v)
        for k, v in this_pass.items():
            pass_means[k].append(statistics.fmean(v))
    assert mismatches == 0, f"{mismatches} timed outputs differ from the persisted Stage 4/5 results: the measurement is invalid"

    # supplementary: the Cross-Encoder over the WHOLE collection
    full = []
    all_texts = [c["text"] for c in CHUNKS]
    ce.predict([(qs[0][1], x) for x in all_texts[:32]], batch_size=32, show_progress_bar=False, activation_fn=torch.nn.Identity())   # warm-up for the larger batch shape
    for _ in range(FULL_COLLECTION_PASSES):
        for _, qq in qs:
            t = time.perf_counter()
            ce.predict([(qq, x) for x in all_texts], batch_size=32, show_progress_bar=False, activation_fn=torch.nn.Identity())
            full.append((time.perf_counter() - t) * 1000.0)
    full_s = summarize_stats(full)
    per_pair_ms = full_s["median_ms"] / len(CHUNKS)
    top10_ce = summarize_stats(samples["ce_score"])
    return dict(
        stage="N7 latency of retrieval and re-ranking (local, single-threaded CPU); LLM latency is reported separately from the recorded calls",
        measured_at_utc=utc_now(), protocol=dict(timer="time.perf_counter (wall clock)", warmup_passes=WARMUP_PASSES, timed_passes=TIMED_PASSES, queries_per_pass=len(qs), samples_per_component=len(qs) * TIMED_PASSES,
                                                  batch_size_top10=K_INIT, batch_size_full_collection=32, percentile="nearest-rank 95th",
                                                  pipeline1="query embedding + FAISS exact Top-10", pipeline2="pipeline 1 + Cross-Encoder scoring of 10 pairs + sort", overhead="pipeline 2 minus pipeline 1 (pair building + Cross-Encoder + sort)"),
        machine_before=info_before, machine_after=dict(load_average_1_5_15=[round(x, 2) for x in os.getloadavg()]),
        one_time_initialisation_not_in_steady_state=init, first_call_of_block_ms={k: round(v, 2) for k, v in first_call.items()},
        steady_state_ms={k: summarize_stats(v) for k, v in samples.items()},
        variability_between_passes=dict(note="coefficient of variation (stdev/mean) of the 20 per-pass means, per component",
                                        cv={k: round(statistics.stdev(v) / statistics.fmean(v), 4) for k, v in pass_means.items()}),
        per_query_median_ms={qid: {k: round(statistics.median(v), 2) for k, v in comp.items() if k in ("pipeline1", "pipeline2", "overhead", "ce_score")} for qid, comp in per_q.items()},
        outputs_identical_to_persisted_results=dict(timed_outputs_checked=len(qs) * TIMED_PASSES, mismatches=mismatches),
        supplementary_cross_encoder_over_entire_collection=dict(
            chunks_scored_per_query=len(CHUNKS), passes=FULL_COLLECTION_PASSES, n=len(full), ms=full_s, median_ms_per_pair=round(per_pair_ms, 3),
            ratio_vs_scoring_only_top10=round(full_s["median_ms"] / top10_ce["median_ms"], 1),
            projections_NOT_MEASURED=dict(basis="linear extrapolation of the measured median per-pair cost; ignores memory effects and batching changes",
                                          seconds_per_query_for_10_000_chunks=round(per_pair_ms * 1e4 / 1000, 1), seconds_per_query_for_1_000_000_chunks=round(per_pair_ms * 1e6 / 1000, 1))),
        what_this_does_not_show="absolute speeds on other hardware, multi-threaded or GPU speed, or LLM latency")


if LATENCY_MODE == "measure":
    LATENCY = measure_latency()
    LATENCY_PATH.write_text(json.dumps(LATENCY, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"measured and saved {LATENCY_PATH.name} ({sha256_file(LATENCY_PATH)[:12]}...)")
elif LATENCY_PATH.exists():
    LATENCY = json.load(open(LATENCY_PATH, encoding="utf-8"))
    print(f"report mode: showing the saved measurement {LATENCY_PATH.name} (sha256 {sha256_file(LATENCY_PATH)[:12]}...), measured {LATENCY['measured_at_utc']}")
else:
    LATENCY = None
    print("report mode: no latency measurement has been recorded yet (set RAG_LATENCY_MODE=measure to take one).")

if LATENCY:
    ss = LATENCY["steady_state_ms"]
    print(f"\nsteady state, single-threaded CPU, n = {LATENCY['protocol']['samples_per_component']} per component (milliseconds)")
    print(f"   {'component':<34} {'mean':>8} {'median':>8} {'stdev':>8} {'p95':>8} {'min':>8} {'max':>8}")
    for key, label in (("embed", "(1) query embedding"), ("search", "(2) FAISS exact Top-10"), ("ce_score", "(3) Cross-Encoder, 10 pairs"), ("sort", "(4) sort 10 logits"),
                       ("pipeline1", "PIPELINE 1  = (1)+(2)"), ("pipeline2", "PIPELINE 2  = (1)+(2)+(3)+(4)"), ("overhead", "re-ranking overhead (P2 - P1)")):
        s = ss[key]
        print(f"   {label:<34} {s['mean_ms']:>8.2f} {s['median_ms']:>8.2f} {s['stdev_ms']:>8.2f} {s['p95_ms']:>8.2f} {s['min_ms']:>8.2f} {s['max_ms']:>8.2f}")
    print(f"\nbetween-pass variability (CV of per-pass means): {LATENCY['variability_between_passes']['cv']}")
    print(f"one-time initialisation (excluded above): {LATENCY['one_time_initialisation_not_in_steady_state']}")
    sup = LATENCY["supplementary_cross_encoder_over_entire_collection"]
    print(f"\nsupplementary: Cross-Encoder over all {sup['chunks_scored_per_query']} chunks per query: median {sup['ms']['median_ms']:.0f} ms (n={sup['n']}), {sup['ratio_vs_scoring_only_top10']}x the cost of scoring only the Top 10; "
          f"{sup['median_ms_per_pair']} ms per pair")
    print(f"   PROJECTION (not measured, linear): {sup['projections_NOT_MEASURED']['seconds_per_query_for_10_000_chunks']} s per query for 10,000 chunks; {sup['projections_NOT_MEASURED']['seconds_per_query_for_1_000_000_chunks']} s per query for 1,000,000 chunks")
    print(f"   timed outputs identical to the persisted Stage 4/5 results: {LATENCY['outputs_identical_to_persisted_results']}")
if STAGE6_RESULT:
    print("\nLLM generation latency from the recorded calls (remote service incl. network; NOT comparable with the local CPU numbers above):", STAGE6_RESULT["latency"])

report mode: showing the saved measurement nist_latency_retrieval.json (sha256 58f335e0f28e...), measured 2026-10-18T11:25:05+00:00

steady state, single-threaded CPU, n = 480 per component (milliseconds)
   component                              mean   median    stdev      p95      min      max
   (1) query embedding                    2.87     2.83     0.28     3.10     2.58     7.84
   (2) FAISS exact Top-10                 0.02     0.02     0.00     0.03     0.02     0.10
   (3) Cross-Encoder, 10 pairs           92.60    93.03    12.34   108.20    47.96   141.68
   (4) sort 10 logits                     0.00     0.00     0.00     0.01     0.00     0.01
   PIPELINE 1  = (1)+(2)                  2.89     2.85     0.28     3.13     2.60     7.87
   PIPELINE 2  = (1)+(2)+(3)+(4)         95.51    95.95    12.40   111.10    50.89   144.52
   re-ranking overhead (P2 - P1)         92.62    93.04    12.34   108.21    47.98   141.70

between-pass variability (CV of per-pass means): {'embed':

## Stage N6i. Stage 6 status for the manifest (one definition, derived from the files on disk)
Used by the final manifest cell below and by `tools/run_stage6_only.py`, which refreshes only these fields without re-executing Stages 1-5.

In [11]:
S6_SUMMARY_TEMPLATE = OUT_DIR / "nist_stage6_human_summary_template.md"
S6_HUMAN_GRADES = OUT_DIR / "nist_stage6_human_grades.csv"             # YOUR graded copy of the review sheet (created by you)
S6_HUMAN_SUMMARY = OUT_DIR / "nist_stage6_human_summary.md"           # written only when the human grades have been entered and tools/summarize_human_grades.py has been run
S6_LIMITS_JSON, S6_LIMITS_MD = OUT_DIR / "nist_stage6_limitations_evidence.json", OUT_DIR / "nist_stage6_limitations_notes.md"


def stage6_manifest_fields():
    smoke_rec = json.load(open(S6_SMOKE, encoding="utf-8")) if S6_SMOKE.exists() else None
    n_answers_ok = len(load_ledger(S6_LEDGER, TASKS_BY_KEY)) if S6_LEDGER.exists() else 0
    n_attempts = len([ln for ln in S6_ATTEMPTS.read_text(encoding="utf-8").splitlines() if ln.strip()]) if S6_ATTEMPTS.exists() else 0
    hf = lambda p: sha256_file(p) if Path(p).exists() else None
    latency_measured = LATENCY_PATH.exists()
    return dict(
        hashes=dict(stage6_prompt_spec_file_sha256=hf(S6_PROMPT_SPEC), stage6_eval_prompts_sha256=EVAL_PROMPTS_SHA256, stage6_settings_sha256=SETTINGS_SHA256,
                    stage6_smoke_test_file_sha256=hf(S6_SMOKE), stage6_answers_ledger_file_sha256=hf(S6_LEDGER), stage6_attempts_log_file_sha256=hf(S6_ATTEMPTS),
                    stage6_answers_file_sha256=hf(S6_ANSWERS), stage6_comparison_csv_sha256=hf(S6_COMPARISON), stage6_human_review_md_sha256=hf(S6_REVIEW_MD),
                    stage6_human_review_csv_sha256=hf(S6_REVIEW_CSV), stage6_grading_guide_file_sha256=hf(S6_GUIDE), stage6_human_summary_template_sha256=hf(S6_SUMMARY_TEMPLATE),
                    stage6_human_grades_file_sha256=hf(S6_HUMAN_GRADES), stage6_human_summary_sha256=hf(S6_HUMAN_SUMMARY), stage6_limitations_evidence_sha256=hf(S6_LIMITS_JSON), stage6_limitations_notes_sha256=hf(S6_LIMITS_MD),
                    latency_retrieval_file_sha256=hf(LATENCY_PATH)),
        llm_status=dict(mode_of_last_notebook_run=STAGE6_MODE, smoke_test_recorded=bool(smoke_rec), smoke_test_passed=bool(smoke_rec and smoke_rec["all_checks_passed"]),
                        evaluation_answers_on_disk=n_answers_ok, expected_answers=len(TASKS), api_attempts_logged_for_evaluation=n_attempts),
        not_run=([] if smoke_rec else ["LLM smoke test (no API call has been made)"]) + ([] if n_answers_ok else ["LLM answer generation (no evaluation answer exists)"]) + ([] if latency_measured else ["latency measurement"]),
        latency_status=("MEASURED: see results/nist_latency_retrieval.json (retrieval and re-ranking) and the per-call latency in results/nist_stage6_answers.json (LLM)" if latency_measured or n_answers_ok
                        else "DEFINED ONLY. No latency has been measured and none is reported here."))


def update_manifest_stage6(path=OUT_DIR / "nist_evaluation_manifest.json"):
    """Refresh only the Stage 6 / latency status fields of an existing manifest (everything else was written by the full notebook run)."""
    m, f = json.load(open(path, encoding="utf-8")), stage6_manifest_fields()
    m["hashes"] = {**{k: v for k, v in m["hashes"].items() if k not in f["hashes"]}, **f["hashes"]}      # same key order as a full run writes
    m["llm_status"], m["not_run"], m["latency_methodology"]["status"] = f["llm_status"], f["not_run"], f["latency_status"]
    Path(path).write_text(json.dumps(m, indent=1, ensure_ascii=False), encoding="utf-8")
    return m


print("Stage 6 status fields (derived from disk):", json.dumps(stage6_manifest_fields()["llm_status"]))

Stage 6 status fields (derived from disk): {"mode_of_last_notebook_run": "off", "smoke_test_recorded": true, "smoke_test_passed": true, "evaluation_answers_on_disk": 48, "expected_answers": 48, "api_attempts_logged_for_evaluation": 48}


## Evaluation manifest and self-test summary
Written last. Statements about the LLM and about latency are **derived from the files that exist on disk**, so the manifest cannot claim that something was run (or not run) when the files say otherwise.

In [ ]:
def versions(*pkgs):
    return {p: md.version(p) for p in pkgs}


S6 = stage6_manifest_fields()

MANIFEST = dict(
    purpose="NIST SP 800-30r1 baseline vs re-ranking evaluation: provenance, frozen rules and methodology",
    primary_metric="preregistered LENIENT rule (any preregistered gold chunk in the Top 3)",
    secondary_analysis="STRICT rule of overlay v1.1 (independently sufficient evidence or complete joint sets); partial evidence is never strict success",
    hashes=dict(
        source_pdf_sha256=PDF_SHA256, corpus_sha256=FROZEN_CORPUS_SHA256, preregistration_sha256=PREREG_SHA256,
        preregistration_file_sha256=sha256_file(OUT_DIR / "nist_preregistration.json"),
        review_sheet_file_sha256=sha256_file(OUT_DIR / "nist_prereg_review_sheet.md"),
        overlay_v1_0_sha256=OVERLAY_V1_SHA256, overlay_v1_1_sha256=OVERLAY_V1_1_SHA256,
        overlay_v1_1_file_sha256=sha256_file(OUT_DIR / "nist_scoring_overlay_v1_1.json"),
        index_file_sha256=INDEX_SHA256, embeddings_sha256=EMB_SHA256,
        baseline_top10_file_sha256=BASELINE_SHA256, rerank_top10_file_sha256=RERANK_SHA256,
        per_query_results_csv_sha256=sha256_file(OUT_DIR / "nist_per_query_results.csv"), prompt_and_generation_config_sha256=PROMPT_SHA256,
        **S6["hashes"]),
    statement_preregistration="The original preregistration (24 questions, gold labels, order, wording) is unchanged; overlay v1.1 is a separate secondary scoring.",
    models=dict(bi_encoder=dict(name=EMBED_MODEL, dim=DIM, max_seq_length=model.max_seq_length, normalised=True),
                cross_encoder=dict(name=CE_NAME, params_millions=round(CE_PARAMS_M, 1), max_seq_length=CE_MAX_LEN, score="raw logit"),
                llm=dict(provider=GEN_PROVIDER, model=GEN_MODEL, request_template=REQUEST_TEMPLATE, note="used only for answer generation; identical for Answer 1 and Answer 2")),
    parameters=dict(k_init=K_INIT, k_final=K_FINAL, index="faiss.IndexFlatIP (exact), unit-length vectors", tie_handling="stable: equal Cross-Encoder scores keep the baseline order",
                    chunking=dict(max_chunk_tokens=MAX_CHUNK_TOKENS, hard_limit_tokens=HARD_LIMIT_TOKENS, entry_max_tokens=ENTRY_MAX_TOKENS, overlap_sentences=OVERLAP_SENTENCES)),
    environment=dict(python=platform.python_version(), machine=platform.machine(), system=platform.system(), omp_num_threads=os.environ.get("OMP_NUM_THREADS"),
                     torch_threads=torch.get_num_threads(), device="cpu", packages=versions("torch", "sentence-transformers", "transformers", "faiss-cpu", "numpy", "pymupdf", "tokenizers", "huggingface_hub")),
    commands=["python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt",
              "open RAG_Baseline_vs_Rerank.ipynb with the .venv kernel and Run All (OMP_NUM_THREADS=1 is set in the first cell; Stage 6 mode defaults to off: no network)",
              "RAG_STAGE6_MODE=smoke    # after putting GROQ_API_KEY in the git-ignored .env: synthetic prompts only",
              "RAG_STAGE6_MODE=evaluate # the 48 evaluation answers; refuses to start without a passing smoke test; resumable",
              "RAG_LATENCY_MODE=measure # retrieval/re-ranking latency (writes results/nist_latency_retrieval.json)",
              "python tools/run_stage6_only.py   # with RAG_STAGE6_MODE=smoke|evaluate: runs ONLY the Stage 6 cells; Stages 1-5 are not re-executed (state restored from the hash-verified files)",
              "python tests/check_frozen_artifacts.py   # verifies every pinned hash and the Stage 6 artifacts without loading a model"],
    latency_methodology=dict(
        status=S6["latency_status"],
        protocol=["time.perf_counter, wall clock, CPU, single-threaded (OMP_NUM_THREADS=1), one query at a time, models already loaded and warmed up (3 discarded warm-up passes)",
                  "baseline retrieval (pipeline 1) = query embedding + FAISS Top-10 search; re-ranking pipeline (pipeline 2) = the same two steps + Cross-Encoder scoring of the ten pairs + sort (cumulative)",
                  "each component also reported separately; 20 timed passes over the 24 questions (n = 480 per component); mean, median, standard deviation, 95th percentile, min, max; index building and model loading excluded",
                  "LLM generation time is the wall clock of each successful HTTP call (remote service, network included), recorded per call; it is never added to the local CPU timings as if the conditions were the same"]),
    llm_status=S6["llm_status"],
    not_run=S6["not_run"],
    frozen_rules="No change to the corpus, queries, gold labels, models or parameters is permitted after observing results; each is protected by a pinned hash.",
    generation=dict(provider=GEN_PROVIDER, model=GEN_MODEL, parameters=GENERATION_CONFIG, provider_request_template=REQUEST_TEMPLATE, settings_sha256=SETTINGS_SHA256,
                    provider_documentation_read=GROQ_DOCS_SNAPSHOT, note="prompt, context format and parameters fixed; identical for both pipelines; the Top-3 context is the only intended difference"),
    self_tests_passed=len(SELF_TESTS))
(OUT_DIR / "nist_evaluation_manifest.json").write_text(json.dumps(MANIFEST, indent=1, ensure_ascii=False), encoding="utf-8")
print("wrote results/nist_evaluation_manifest.json")
print(f"not run: {S6['not_run'] or 'nothing'}")
print(f"\nALL {len(SELF_TESTS)} SELF-TESTS PASSED:")
for t in SELF_TESTS: print("   -", t)